# Lib Installation

In [ ]:
!pip install  lightning pytorch-lightning torchvision  tqdm python-dotenv PyYAML wilds

# Config

In [ ]:
config = {
    "algo": "FOND",
    "dataset": ["PACS", "VLCS", "OfficeHome", "Wilds"],  # Options: VLCS, PACS
    "test_set_id": 0,  # 0, 1, 2, or 3

    # Paths
    "data_dir": {
        "PACS": "/kaggle/input/pacs-dataset/kfold",
        "VLCS": "/kaggle/input/vlcsdataset/",
        "OfficeHome": "/kaggle/input/officehome/OfficeHome/",
        "Wilds": ""
    },
    "log_dir": "./logs",

    # Training settings
    "n_epochs": 5001,
    "checkpoint_freq": 300,
    "holdout_fraction": 0.2,

    # Model checkpointing
    "model_checkpoint": {
        "metric": "val/oacc",
        "maximize": True
    },

    # Seeds
    "overall_seed": 1,
    "trial_id": 0,
    "hparam_id": 0,  # 0=default hparams, 1-4=random variants

    # System
    "num_workers": 4,

    # Dataset config
    "overlap": "high",
    "num_classes": None,
    "num_domain_linked_classes": None,

    # AutoFOND specific
    "auto_augment": True,
    "augment_search_epochs": 5,
    "augment_policy_size": 5,
    "augment_num_policies": 3,

    # MixStyle (feature-level augmentation)
    "mixstyle_p": 0.5,        # probability to apply MixStyle
    "mixstyle_alpha": 0.1,    # beta distribution alpha parameter

    # SWAD
    "use_swad": True,
    "swad_start_epoch": 1,

    # Teacher paths (empty for regular FOND)
    "teacher_paths": {}
}


# Imports

In [ ]:
import math
from typing import List, Any, Dict, Optional, Callable, Tuple
import argparse
import logging
import hashlib
import numpy as np
from collections import Counter
import collections
import time
from datetime import datetime
import lightning as L
from tqdm import tqdm
import csv
import json
from pathlib import Path
import os
import torch
import torchmetrics
import torchvision.datasets.folder
from PIL import Image, ImageFile
from torch.utils.data import ConcatDataset, Dataset, Subset, TensorDataset
from torchvision import transforms
from torchvision.datasets import MNIST, ImageFolder
from torchvision.transforms.functional import rotate
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models
from wilds.datasets.camelyon17_dataset import Camelyon17Dataset





# SRC

## Domain Creation

In [ ]:



def create_domains(
    num_classes: int, num_linked: int, num_train_domains: int
) -> List[List[int]]:
    """
    Determine and distribute domain-linked and domain-shared classes.
    Domain-linked classes will come from the same domain, i.e., idx=0.
    Domain-shared classes will exist in each of the remaining domains

    Args:
        num_classes: number of overall classes within all the domains
        num_linked: number of classes that are linked to an individual domain
        num_train_domains: number of different training domains
    """
    assert num_linked <= num_classes
    domain_shared = [i for i in range(num_linked, num_classes)]
    domain_linked = [i for i in range(num_linked)]
    logging.info(f"Domain-shared classes: {domain_shared}")
    logging.info(f"Domain-linked classes: {domain_linked}")
    # domains = [domain_shared.copy() for i in range(num_train_domains)]
    domains = [[] for i in range(num_train_domains)]

    # first domain is domain-linked only
    domains[0] = domain_linked
    # other domains contain domain-shared only
    domains[1:] = [domain_shared.copy() for i in range(1, num_train_domains)]

    logging.info(f"domains[0]: {domains[0]}")
    logging.info(f"domains[1:]: {domains[1:]}")

    return domains


def create_domains_1(
    num_classes: int, num_linked: int, num_domains: int
) -> List[List[int]]:
    """
    Args:
        num_classes: number of overall classes within all the domains
        num_linked: number of classes that are linked to an individual domain
        num_domains: number of different domains, including test domain
    """
    assert num_linked < num_classes
    domain_shared = [i for i in range(num_linked, num_classes)]
    print(f"Shared classes: {domain_shared}")
    num_train_domains = num_domains - 1
    domains = [domain_shared.copy() for i in range(num_train_domains)]

    for class_idx in range(num_linked):
        domain_idx = class_idx % num_train_domains
        domains[domain_idx].append(class_idx)
    return domains


def create_domains_2(
    num_classes: int, num_linked_ratio: float, num_domains: int
) -> List[List[int]]:
    """
    Args:
        num_classes: number of overall classes within all the domains
        num_linked_ratio: ratio of linked classes to the total number of classes
        num_domains: number of different domains, including test domain
    """
    num_linked = math.floor(num_linked_ratio * num_classes)
    return create_domains_1(num_classes, num_linked, num_domains)



# Testing
create_domains(num_classes=10, num_linked=5, num_train_domains=3)
# Testing domain linked only
create_domains(num_classes=10, num_linked=10, num_train_domains=3)

# Testing same classes with different overlap
domains = create_domains_1(num_classes=10, num_linked=3, num_domains=4)
print(domains)
domains = create_domains_1(num_classes=10, num_linked=5, num_domains=4)
print(domains)

# Testing different classes with same overlap percentage
domains = create_domains_2(num_classes=10, num_linked_ratio=0.2, num_domains=4)
print(domains)
domains = create_domains_2(num_classes=5, num_linked_ratio=0.2, num_domains=4)
print(domains)


## Hparams

In [ ]:
"""
Hyper-parameter registry (UPDATED WITH MIXSTYLE)
"""

def seed_hash(*args):
    args_str = str(args)
    return int(hashlib.md5(args_str.encode("utf-8")).hexdigest(), 16) % (2**31)

def _define_hparam(hparams, hparam_name, default_val, random_val_fn):
    hparams[hparam_name] = (hparams, hparam_name, default_val, random_val_fn)

def _hparams(algorithm, dataset, random_seed):
    SMALL_IMAGES = ["Debug28", "RotatedMNIST", "ColoredMNIST"]

    hparams = {}

    def _hparam(name, default_val, random_val_fn):
        assert name not in hparams
        random_state = np.random.RandomState(seed_hash(random_seed, name))
        hparams[name] = (default_val, random_val_fn(random_state))

    # -----------------------------
    # GENERAL HPARAMS
    # -----------------------------
    _hparam("data_augmentation", True, lambda r: True)
    _hparam("resnet18", True, lambda r: True)
    _hparam("resnet_dropout", 0.0, lambda r: r.choice([0.0, 0.1, 0.5]))
    _hparam("class_balanced", False, lambda r: False)
    _hparam("nonlinear_classifier", False, lambda r: False)

    # -----------------------------
    # 🔥 MIXSTYLE HPARAMS (added)
    # -----------------------------
    _hparam("mixstyle_p", 0.5, lambda r: 0.5)
    _hparam("mixstyle_alpha", 0.1, lambda r: 0.1)

    # -----------------------------
    # ALGORITHM-SPECIFIC
    # -----------------------------
    if algorithm in ["FOND", "FOND_NC", "FOND_N", "NOC"]:
        _hparam("temperature", 0.07, lambda r: 0.07 * r.uniform(0.75, 1.25))
        _hparam("base_temperature", 0.07, lambda r: 0.07)
        _hparam("xdom_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("error_lmbd", 1, lambda r: 10 ** r.uniform(-1, 3))
        _hparam("xda_alpha", 1, lambda r: 10 ** r.uniform(0, 1))
        _hparam("xda_beta", 1, lambda r: 10 ** r.uniform(0, 1))

    # -----------------------------
    # DATASET & ALGO SPECIFIC
    # -----------------------------
    if dataset in SMALL_IMAGES:
        _hparam("lr", 1e-3, lambda r: 10 ** r.uniform(-4.5, -2.5))
    else:
        _hparam("lr", 5e-5, lambda r: 10 ** r.uniform(-5, -3.5))

    if dataset in SMALL_IMAGES:
        _hparam("weight_decay", 0.0, lambda r: 0.0)
    else:
        _hparam("weight_decay", 0.0, lambda r: 10 ** r.uniform(-6, -2))

    if dataset in SMALL_IMAGES:
        _hparam("batch_size", 64, lambda r: int(2 ** r.uniform(3, 9)))
    elif algorithm == "ARM":
        _hparam("batch_size", 8, lambda r: 8)
    elif dataset == "DomainNet":
        _hparam("batch_size", 32, lambda r: int(2 ** r.uniform(3, 5)))
    else:
        _hparam("batch_size", 32, lambda r: int(2 ** r.uniform(3, 5.5)))

    return hparams

def default_hparams(algorithm, dataset):
    return {a: b for a, (b, c) in _hparams(algorithm, dataset, 0).items()}

def random_hparams(algorithm, dataset, seed):
    return {a: c for a, (b, c) in _hparams(algorithm, dataset, seed).items()}


## Misc

In [ ]:

class _SplitDataset(torch.utils.data.Dataset):
    """Used by split_dataset"""

    def __init__(self, underlying_dataset, keys):
        super(_SplitDataset, self).__init__()
        self.underlying_dataset = underlying_dataset
        self.keys = keys

    def __getitem__(self, key):
        return self.underlying_dataset[self.keys[key]]

    def __len__(self):
        return len(self.keys)


def split_dataset(dataset, n, seed=0):
    """
    Return a pair of datasets corresponding to a random split of the given
    dataset, with n datapoints in the first dataset and the rest in the last,
    using the given random seed
    """
    assert n <= len(dataset)
    keys = list(range(len(dataset)))
    np.random.RandomState(seed).shuffle(keys)
    keys_1 = keys[:n]
    keys_2 = keys[n:]
    return _SplitDataset(dataset, keys_1), _SplitDataset(dataset, keys_2)


def seed_hash(*args):
    """
    Derive an integer hash from all args, for use as a random seed.
    """
    args_str = str(args)
    return int(hashlib.md5(args_str.encode("utf-8")).hexdigest(), 16) % (2**31)


def make_weights_for_balanced_classes(dataset):
    counts = Counter()
    classes = []
    for _, y in dataset:
        y = int(y)
        counts[y] += 1
        classes.append(y)

    n_classes = len(counts)

    weight_per_class = {}
    for y in counts:
        weight_per_class[y] = 1 / (counts[y] * n_classes)

    weights = torch.zeros(len(dataset))
    for i, y in enumerate(classes):
        weights[i] = weight_per_class[int(y)]

    return weights
def compute_loss(network, loader, device):
    """
    Compute average loss for a given loader.
    Returns: average loss value
    """
    total_loss = 0.0
    num_batches = 0

    network.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            p = network.predict(x)

            # Compute loss
            loss = torch.nn.functional.cross_entropy(p, y, reduction='mean')
            total_loss += loss.item()
            num_batches += 1

    network.train()

    avg_loss = total_loss / num_batches if num_batches > 0 else 0.0
    return avg_loss

def accuracy(network, loader, weights, device, dataset):
    correct = 0
    total = 0
    weights_offset = 0
    overlapping_classes = dataset.overlapping_classes
    num_classes = dataset.num_classes

    f1_score = torchmetrics.F1Score(
        task="multiclass", num_classes=num_classes, average="macro"
    ).to(device)
    per_class_accuracy = torchmetrics.Accuracy(
        task="multiclass",
        num_classes=num_classes,
        average=None,
    ).to(device)
    accuracy = torchmetrics.Accuracy(
        task="multiclass",
        num_classes=num_classes,
        average="macro",
    ).to(device)
    recall = torchmetrics.Recall(
        task="multiclass",
        num_classes=num_classes,
        average="macro",
    ).to(device)
    precision = torchmetrics.Precision(
        task='multiclass',  # ✅ Fixed typo
        num_classes=num_classes,
        average='macro'
    ).to(device)
    conf_mat = torchmetrics.ConfusionMatrix(
        task="multiclass",
        num_classes=num_classes
    ).to(device)  # ✅ Added .to(device)

    network.eval()
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            p = network.predict(x)

            if weights is None:
                batch_weights = torch.ones(len(x))
            else:
                batch_weights = weights[weights_offset : weights_offset + len(x)]
                weights_offset += len(x)
            batch_weights = batch_weights.to(device)

            if p.size(1) == 1:
                correct += (
                    (p.gt(0).eq(y).float() * batch_weights.view(-1, 1)).sum().item()
                )
            else:
                correct += (p.argmax(1).eq(y).float() * batch_weights).sum().item()
            total += batch_weights.sum().item()

            # update metrics
            accuracy.update(p, y)
            recall.update(p, y)
            precision.update(p, y)
            f1_score.update(p, y)
            per_class_accuracy.update(p, y)
            conf_mat.update(p, y)

    network.train()
    compute_acc = accuracy.compute().item()
    compute_recall = recall.compute().item()
    compute_precision = precision.compute().item()  # ✅ Fixed typo
    compute_f1 = f1_score.compute().item()
    compute_per_class_acc = per_class_accuracy.compute().cpu().numpy()
    cm_compute = conf_mat.compute().cpu().numpy()  # ✅ Convert to numpy

    overlap_class_acc = []
    non_overlap_class_acc = []
    per_class_acc_dict = {}
    for i in range(num_classes):
        per_class_acc_dict[i] = float(compute_per_class_acc[i])
        if i in overlapping_classes:
            overlap_class_acc.append(compute_per_class_acc[i])
        else:
            non_overlap_class_acc.append(compute_per_class_acc[i])

    if len(non_overlap_class_acc) == 0:
        non_overlap_class_acc = -1
    else:
        non_overlap_class_acc = np.mean(non_overlap_class_acc)
    if len(overlap_class_acc) == 0:
        overlap_class_acc = -1
    else:
        overlap_class_acc = np.mean(overlap_class_acc)

    other_acc = correct / total

    return (
        float(compute_acc),
        float(compute_recall),
        float(compute_f1),
        float(compute_precision),      # ✅ Moved before oacc/nacc
        float(overlap_class_acc),
        float(non_overlap_class_acc),
        per_class_acc_dict,
        cm_compute
    )
class _InfiniteSampler(torch.utils.data.Sampler):
    """Wraps another Sampler to yield an infinite stream."""

    def __init__(self, sampler):
        self.sampler = sampler

    def __iter__(self):
        while True:
            for batch in self.sampler:
                yield batch


class InfiniteDataLoader:
    def __init__(self, dataset, weights, batch_size, num_workers):
        super().__init__()

        if weights is not None:
            sampler = torch.utils.data.WeightedRandomSampler(
                weights, replacement=True, num_samples=batch_size
            )
        else:
            sampler = torch.utils.data.RandomSampler(dataset, replacement=True)

        if weights == None:
            weights = torch.ones(len(dataset))

        batch_sampler = torch.utils.data.BatchSampler(
            sampler, batch_size=batch_size, drop_last=True
        )

        self._infinite_iterator = iter(
            torch.utils.data.DataLoader(
                dataset,
                num_workers=num_workers,
                batch_sampler=_InfiniteSampler(batch_sampler),
            )
        )

    def __iter__(self):
        while True:
            yield next(self._infinite_iterator)

    def __len__(self):
        raise ValueError


class FastDataLoader:
    """DataLoader wrapper with slightly improved speed by not respawning worker
    processes at every epoch."""

    def __init__(self, dataset, batch_size, num_workers):
        super().__init__()

        batch_sampler = torch.utils.data.BatchSampler(
            torch.utils.data.RandomSampler(dataset, replacement=False),
            batch_size=batch_size,
            drop_last=False,
        )

        self._infinite_iterator = iter(
            torch.utils.data.DataLoader(
                dataset,
                num_workers=num_workers,
                batch_sampler=_InfiniteSampler(batch_sampler),
            )
        )

        self._length = len(batch_sampler)

    def __iter__(self):
        for _ in range(len(self)):
            yield next(self._infinite_iterator)

    def __len__(self):
        return self._length


def config_logging():
    """
    Reusable code for formatting the logger
    """
    logging.basicConfig(
        format="%(asctime)s,%(msecs)03d %(levelname)-8s [%(filename)s:%(funcName)s:%(lineno)d] %(message)s",
        datefmt="%Y-%m-%d:%H:%M:%S",
        level=logging.INFO,
    )


## Simple Logger

In [ ]:
"""
Simple CSV logger to replace W&B during development
Keeps the same interface for easy swap later
"""
class CSVLogger:
    """Lightweight logger that writes metrics to CSV"""

    def __init__(self, csv_path: str, root_dir:str=None):
        self.csv_path = Path(csv_path)
        self.csv_path.parent.mkdir(parents=True, exist_ok=True)
        self.fieldnames = None
        self.file = None
        self.writer = None
        self._init_csv()
        self.root = root_dir if root_dir is not None else csv_path.parent
    def _init_csv(self):
        """Initialize CSV file with headers"""
        self.file = open(self.csv_path, 'w', newline='')
        self.writer = None  # Will be created on first log
    def return_root(self):
        return self.root
    def log(self, metrics: Dict[str, Any], step: Optional[int] = None):
        """
        Log metrics to CSV

        Args:
            metrics: Dictionary of metric_name -> value
            step: Training step (optional, will be added if provided)
        """
        if step is not None:
            metrics = {"step": step, **metrics}

        # Flatten nested dictionaries (e.g., {"train/acc": 0.9})
        flat_metrics = {}
        for key, value in metrics.items():
            if isinstance(value, dict):
                for subkey, subvalue in value.items():
                    flat_metrics[f"{key}/{subkey}"] = subvalue
            else:
                flat_metrics[key] = value

        # Initialize writer with fieldnames on first call
        if self.writer is None:
            self.fieldnames = list(flat_metrics.keys())
            self.writer = csv.DictWriter(self.file, fieldnames=self.fieldnames)
            self.writer.writeheader()

        # Add new fields if they appear
        new_fields = set(flat_metrics.keys()) - set(self.fieldnames)
        if new_fields:
            self.fieldnames.extend(sorted(new_fields))
            # Rewrite file with new headers
            self.file.close()
            self._rewrite_with_new_fields(flat_metrics)
            return

        self.writer.writerow(flat_metrics)
        self.file.flush()  # Ensure immediate write

    def _rewrite_with_new_fields(self, new_row: Dict[str, Any]):
        """Rewrite CSV when new fields are discovered"""
        # Read existing rows
        with open(self.csv_path, 'r') as f:
            reader = csv.DictReader(f)
            existing_rows = list(reader)

        # Rewrite with updated fieldnames
        self.file = open(self.csv_path, 'w', newline='')
        self.writer = csv.DictWriter(self.file, fieldnames=self.fieldnames)
        self.writer.writeheader()
        for row in existing_rows:
            self.writer.writerow(row)
        self.writer.writerow(new_row)
        self.file.flush()

    def save_config(self, config: Dict[str, Any], path: Optional[str] = None):
        """Save experiment config as JSON"""
        if path is None:
            path = self.csv_path.parent / "config.json"
        with open(path, 'w') as f:
            json.dump(config, f, indent=2)

    def close(self):
        """Close CSV file"""
        if self.file:
            self.file.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()


class PrintLogger:
    """Even simpler logger that just prints to console"""

    def log(self, metrics: Dict[str, Any], step: Optional[int] = None):
        step_str = f"[Step {step}] " if step is not None else ""
        metric_str = ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                               for k, v in metrics.items())
        print(f"{step_str}{metric_str}")

    def save_config(self, config: Dict[str, Any], path: Optional[str] = None):
        print(f"Config: {json.dumps(config, indent=2)}")

    def close(self):
        pass


# Dataset

## Base

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True

DATASETS2 = [
    # Debug
    "Debug28",
    "Debug224",
    # Small images
    "ColoredMNIST",
    "RotatedMNIST",
    # Big images
    "VLCS",
    "PACS",
    "OfficeHome",
    "TerraIncognita",
    "DomainNet",
    "SVIRO",
    # WILDS datasets
    "WILDSCamelyon",
    "WILDSFMoW",
    # Spawrious datasets
    "SpawriousO2O_easy",
    "SpawriousO2O_medium",
    "SpawriousO2O_hard",
    "SpawriousM2M_easy",
    "SpawriousM2M_medium",
    "SpawriousM2M_hard",
]

# OVERLAP_TYPES = ["none", "low", "mid", "high", "full", "0", "33", "66", "100"]
SPECIAL_OVERLAP_TYPES = ["0", "low", "high", "low_linked_only", "high_linked_only"]
OVERLAP_TYPES = ["0", "low", "high", "low_linked_only", "high_linked_only"]


def get_domain_classes(N_c, N_oc, repeat, N_s, seed):
    N_noc = N_c - N_oc
    Q = []
    C = list(range(N_c))

    random_state = np.random.RandomState(seed)

    # choose non overlapping classes
    C_noc = list(random_state.choice(C, replace=False, size=N_noc))
    C_oc = [x for x in C if x not in C_noc]

    # add to queue
    Q.extend(C_noc + list(np.repeat(C_oc, repeat)))

    # Round-robing distribution of classes
    domain_classes = [Q[i::N_s] for i in range(N_s)]

    # assert overlapping classes
    overlap = np.zeros(N_c)
    for cls_list in domain_classes:
        np.add.at(overlap, cls_list, 1)

    assert C_oc == list(np.where(overlap > 1)[0])

    # output
    print("C_noc", C_noc)
    print("C_oc", C_oc)
    print("Q", Q)
    print("domain_classes", domain_classes)

    return domain_classes


class DomainBedImageFolder(ImageFolder):
    """
    Custom class to allow class filtering
    """

    def __init__(
        self,
        root: str,
        transform: Optional[Callable] = None,
        target_transform: Optional[Callable] = None,
        remove_classes: List[int] = [],
        allowed_classes: List[int] = [],
        is_test_env: Optional[bool] = None,
        env_name: Optional[str] = None,
    ):
        super().__init__(root, transform, target_transform)

        # Remove specified classes
        old_samples = self.samples
        self.samples = []
        self.targets = []
        self.is_test_env = is_test_env
        self.allowed_classes = allowed_classes
        self.remove_classes = remove_classes
        self.env_name = env_name
        for sample in old_samples:
            _, target = sample

            if target not in remove_classes:
                self.samples.append(sample)
                self.targets.append(target)

        self.imgs = self.samples
        self.classes = list(set(self.targets))

    def __len__(self) -> int:
        return len(self.samples)

    def __str__(self):
        return (
            f"{self.env_name}: is_test_env={self.is_test_env}, allowed_classes={self.allowed_classes},"
            f" list(set(self.targets))={self.classes}, num_samples={len(self)}"
        )


def get_overlapping_classes(
    class_split: List[List[int]], num_classes: int
) -> List[int]:
    """
    Return the classes in multiple domains.
    """
    overlap = np.zeros(num_classes)
    for data in class_split:
        np.add.at(overlap, data, 1)

    overlapping_classes = list(np.where(overlap > 1)[0])

    return overlapping_classes


def get_dataset_class(dataset_name):
    """Return the dataset class with the given name."""
    if dataset_name not in globals():
        raise NotImplementedError("Dataset not found: {}".format(dataset_name))
    return globals()[dataset_name]


def num_environments(dataset_name):
    return len(get_dataset_class(dataset_name).ENVIRONMENTS)


class MultipleDomainDataset:
    N_STEPS = 5001  # Default, subclasses may override
    CHECKPOINT_FREQ = 100  # Default, subclasses may override
    N_WORKERS = 8  # Default, subclasses may override
    ENVIRONMENTS = None  # Subclasses should override
    INPUT_SHAPE = None  # Subclasses should override

    def __getitem__(self, index):
        return self.datasets[index]

    def __len__(self):
        return len(self.datasets)


class Debug(MultipleDomainDataset):
    def __init__(self, root, test_envs, hparams):
        super().__init__()
        self.input_shape = self.INPUT_SHAPE
        self.num_classes = 2
        self.datasets = []
        for _ in [0, 1, 2]:
            self.datasets.append(
                TensorDataset(
                    torch.randn(16, *self.INPUT_SHAPE),
                    torch.randint(0, self.num_classes, (16,)),
                )
            )


class Debug28(Debug):
    INPUT_SHAPE = (3, 28, 28)
    ENVIRONMENTS = ["0", "1", "2"]


class Debug224(Debug):
    INPUT_SHAPE = (3, 224, 224)
    ENVIRONMENTS = ["0", "1", "2"]


class MultipleEnvironmentMNIST(MultipleDomainDataset):
    def __init__(
        self,
        root,
        environments,
        dataset_transform,
        input_shape,
        num_classes,
        test_envs: List[int],
        domain_class_filter: List[List[int]],
    ):
        super().__init__()
        if root is None:
            raise ValueError("Data directory not specified!")

        original_dataset_tr = MNIST(root, train=True, download=True)
        original_dataset_te = MNIST(root, train=False, download=True)

        original_images = torch.cat(
            (original_dataset_tr.data, original_dataset_te.data)
        )

        original_labels = torch.cat(
            (original_dataset_tr.targets, original_dataset_te.targets)
        )

        shuffle = torch.randperm(len(original_images))

        original_images = original_images[shuffle]
        original_labels = original_labels[shuffle]

        assert len(test_envs) == 1, "Not performing leave-one-domain-out validation"
        num_envs = len(environments)

        self.num_classes = num_classes
        self.overlapping_classes = get_overlapping_classes(
            domain_class_filter, self.num_classes
        )

        # Dynamically associate a filter with a domain except for test_envs[0]
        num_filters = len(domain_class_filter)
        assert num_envs - 1 == num_filters  # b/c exempt first test env
        shift_filter = list(range(num_filters)) + list(range(num_filters))
        shift_filter = shift_filter[test_envs[0] : test_envs[0] + num_filters]

        self.datasets = []

        for i in range(len(environments)):
            images = original_images[i :: len(environments)]
            labels = original_labels[i :: len(environments)]
            self.datasets.append(dataset_transform(images, labels, environments[i]))

        self.input_shape = input_shape


class MultipleEnvironmentImageFolder(MultipleDomainDataset):
    def __init__(
        self,
        root,
        test_envs,
        augment,
        hparams,
        domain_class_filter: List[List[int]],
        num_classes=None,
    ):
        super().__init__()
        environments = [f.name for f in os.scandir(root) if f.is_dir()]
        environments = sorted(environments)
        num_envs = len(environments)

        assert len(test_envs) == 1, "Not performing leave-one-domain-out validation"

        self.idx_to_class = self.get_idx_to_class(
            os.path.join(root, environments[test_envs[0]])
        )
        self.num_classes = (
            len(self.idx_to_class) if num_classes is None else num_classes
        )

        self.overlapping_classes = get_overlapping_classes(
            domain_class_filter, self.num_classes
        )
        logging.info(f"Overlapping classes: {self.overlapping_classes}")

        # Dynamically associate a filter with a domain except for test_envs[0]
        num_filters = len(domain_class_filter)
        print(f"num_envs={num_envs}, num_filters={num_filters}, domain_class_filter={domain_class_filter}")
        assert num_envs - 1 == num_filters  # b/c exempt first test env
        shift_filter = list(range(num_filters)) + list(range(num_filters))
        shift_filter = shift_filter[test_envs[0] : test_envs[0] + num_filters]

        transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        augment_transform = transforms.Compose(
            [
                # transforms.Resize((224,224)),
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
                transforms.RandomGrayscale(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        self.datasets = []
        for i, environment in enumerate(environments):
            path = os.path.join(root, environment)

            # setup augmentation
            if augment and (i not in test_envs):
                env_transform = augment_transform
            else:
                env_transform = transform

            # setup class filtering
            all_classes = set(list(self.idx_to_class.keys()))
            if i not in test_envs:
                filter = domain_class_filter[shift_filter.pop()]
                if filter == []:
                    continue
                remove_classes = list(all_classes - set(filter))

                env_dataset = DomainBedImageFolder(
                    path,
                    transform=env_transform,
                    remove_classes=remove_classes,
                    is_test_env=False,
                    allowed_classes=filter,
                    env_name=environment,
                )
            else:
                remove_classes = list(range(self.num_classes, len(all_classes)))
                env_dataset = DomainBedImageFolder(
                    path,
                    transform=env_transform,
                    remove_classes=remove_classes,
                    is_test_env=True,
                    allowed_classes=list(range(self.num_classes)),
                    env_name=environment,
                )
                assert self.num_classes == len(env_dataset.classes)

            # print(f"\n[info] environment: {env_dataset.env_name}, classes: {env_dataset.allowed_classes}, is_test: {env_dataset.is_test_env}")
            logging.info(f"Created domain -> {env_dataset}")
            self.datasets.append(env_dataset)

        self.input_shape = (
            3,
            224,
            224,
        )

    def get_overlapping_classes(
        self, class_split: List[List[int]], num_classes: int
    ) -> List[int]:
        """
        Return the classes in multiple domains.
        """
        overlap = np.zeros(num_classes)
        for data in class_split:
            np.add.at(overlap, data, 1)

        overlapping_classes = list(np.where(overlap > 1)[0])

        return overlapping_classes

    def get_idx_to_class(self, data_dir: str) -> Dict[int, str]:
        dataset = ImageFolder(data_dir)
        idx_to_class = {}
        for key, value in dataset.class_to_idx.items():
            idx_to_class.update({value: key})

        assert len(dataset.class_to_idx) == len(
            idx_to_class
        ), "Class and labels are not one-to-one"

        return idx_to_class

## PACS

In [ ]:


class PACS(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["A", "C", "P", "S"])
    NUM_CLASSES = 7
    OVERLAP_CONFIG = {
        "0": [[0, 1], [2, 3], [4, 5, 6]],
        "low": [[0, 1, 2], [2, 3, 4], [4, 5, 6]],
        "high": [[0, 1, 2, 3], [2, 3, 4, 5], [4, 5, 6, 0]],
        "100": [list(range(7)), list(range(7)), list(range(7))],
        "low_linked_only": [[0, 1], [3], [5, 6]],
        "high_linked_only": [[0, 1], [], [6]],
    }

    # overlap_type
    def __init__(
        self,
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None
    ):
        # print(f"[info] {type(self)}, test_envs: {test_envs}, overlap: {class_overlap_id}")
        self.dir = root
        print(root)
        self._num_source_domains = 3
        self._num_classes = 7

        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            PACS.OVERLAP_CONFIG[overlap_type],
        )


## VLCS

In [ ]:
class VLCS(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["C", "L", "S", "V"])
    NUM_CLASSES = 5
    OVERLAP_CONFIG = {
        "0": [[0, 1], [2, 3], [4]],
        "low": [[0, 1, 2], [2, 3], [3, 4]],
        "high": [[0, 1, 2], [2, 3, 4], [3, 4, 0]],
        "low_linked_only": [[0, 1], [], [4]],
        "high_linked_only": [[1], [], []],
        "100": [list(range(5)), list(range(5)), list(range(5))],
    }

    def __init__(
        self,
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None
    ):
        # print(f"[info] {type(self)}, test_envs: {test_envs}, overlap: {class_overlap_id}")
        self.dir = os.path.join(root, "VLCS/")
        self._num_source_domains = 3
        self._num_classes = 5

        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            VLCS.OVERLAP_CONFIG[overlap_type],
        )


## OfficeHome

In [ ]:


class OfficeHome(MultipleEnvironmentImageFolder):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = sorted(["A", "C", "P", "R"])
    OVERLAP_CONFIG = {
        "0": [list(range(0, 22)), list(range(22, 44)), list(range(44, 65))],
        "low": [list(range(0, 30)), list(range(14, 44)), list(range(35, 65))],  # 25/65
        "high": [list(range(0, 38)), list(range(5, 44)), list(range(27, 65))],  # 50/65
        "low_linked_only": [
            list(range(0, 14)),
            list(range(30, 35)),
            list(range(44, 65)),
        ],
        "high_linked_only": [list(range(0, 5)), [], list(range(44, 65))],
        "100": [list(range(65)), list(range(65)), list(range(65))],
    }

    def __init__(
        self,
        root: str,
        test_envs: List[int],
        hparams: dict,
        num_classes: int,
        num_domain_linked_classes: int,
        overlap_type=None,
        overlap_seed=None,
    ):

        self.dir = root
        self._num_source_domains = 3
        self._num_classes = 65

        domain_class_filter = []
        if overlap_type is not None:
            logging.info(
                f"Using predefined class distributions: overlap_type={overlap_type}"
            )
            domain_class_filter = OfficeHome.OVERLAP_CONFIG[overlap_type]
        else:
            logging.info(
                f"Using dynamic class distributions: num_classes={num_classes}, num_linked={num_domain_linked_classes}, num_train_domains={self._num_source_domains}"
            )
            assert num_classes <= self._num_classes
            self._num_classes = num_classes
            domain_class_filter = create_domains(
                num_classes=num_classes,
                num_linked=num_domain_linked_classes,
                num_train_domains=self._num_source_domains,
            )
        print(f"DEBUG: overlap_type={overlap_type}")
        print(f"DEBUG: domain_class_filter={domain_class_filter}")
        print(f"DEBUG: len(domain_class_filter)={len(domain_class_filter)}")
        print(f"DEBUG: test_envs={test_envs}")


        super().__init__(
            self.dir,
            test_envs,
            hparams["data_augmentation"],
            hparams,
            domain_class_filter=domain_class_filter,
            num_classes=self._num_classes
        )


## Wilds

In [ ]:

class WILDSEnvironment:
    def __init__(
        self,
        wilds_dataset,
        metadata_name,
        metadata_value,
        transform=None,
        is_test_env=False,
        allowed_classes=None  # Filter samples by class
    ):
        self.name = metadata_name + "_" + str(metadata_value)
        self.is_test_env = is_test_env

        metadata_index = wilds_dataset.metadata_fields.index(metadata_name)
        metadata_array = wilds_dataset.metadata_array
        subset_indices = torch.where(
            metadata_array[:, metadata_index] == metadata_value
        )[0]

        # Filter by allowed classes if specified
        if allowed_classes is not None:
            y_values = wilds_dataset.y_array[subset_indices]
            class_mask = torch.zeros(len(subset_indices), dtype=torch.bool)
            for allowed_class in allowed_classes:
                class_mask |= (y_values == allowed_class)
            subset_indices = subset_indices[class_mask]
            print(f"  {self.name}: Filtered to classes {allowed_classes}, {len(subset_indices)} samples")
        else:
            print(f"  {self.name}: All classes, {len(subset_indices)} samples")

        self.dataset = wilds_dataset
        self.indices = subset_indices
        self.transform = transform

    def __getitem__(self, i):
        x = self.dataset.get_input(self.indices[i])
        if type(x).__name__ != "Image":
            x = Image.fromarray(x)

        y = self.dataset.y_array[self.indices[i]]
        if self.transform is not None:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.indices)


class WILDSDataset(MultipleDomainDataset):
    INPUT_SHAPE = (3, 224, 224)

    def __init__(
        self,
        dataset,
        metadata_name,
        test_envs,
        augment,
        hparams,
        overlap_config
    ):
        super().__init__()

        transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        augment_transform = transforms.Compose(
            [
                transforms.Resize((224, 224)),
                transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ColorJitter(0.3, 0.3, 0.3, 0.3),
                transforms.RandomGrayscale(),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
                ),
            ]
        )

        self.datasets = []
        metadata_values = self.metadata_values(dataset, metadata_name)

        print(f"[WILDS] Creating {len(metadata_values)} environments with overlap config: {overlap_config}")

        # Map environment index to allowed classes
        # Test environments get all classes, source environments get restricted classes
        source_env_idx = 0
        for i, metadata_value in enumerate(metadata_values):
            if augment and (i not in test_envs):
                env_transform = augment_transform
            else:
                env_transform = transform

            # Determine allowed classes for this environment
            if i in test_envs:
                # Test environment gets all classes
                allowed_classes = None
                print(f"[WILDS] Env {i} (TEST): all classes")
            else:
                # Source environment gets restricted classes from overlap_config
                if source_env_idx < len(overlap_config):
                    allowed_classes = overlap_config[source_env_idx]
                    print(f"[WILDS] Env {i} (SOURCE): classes {allowed_classes}")
                else:
                    allowed_classes = None
                    print(f"[WILDS] Env {i} (SOURCE): all classes (no config)")
                source_env_idx += 1

            env_dataset = WILDSEnvironment(
                dataset,
                metadata_name,
                metadata_value,
                env_transform,
                is_test_env=(i in test_envs),
                allowed_classes=allowed_classes
            )

            self.datasets.append(env_dataset)

        self.input_shape = (3, 224, 224)
        self.num_classes = dataset.n_classes

        # Set overlapping classes from overlap_config (same as PACS/VLCS)
        all_overlapping = set()
        for domain_classes in overlap_config:
            all_overlapping.update(domain_classes)
        self.overlapping_classes = sorted(list(all_overlapping))

        print(f"[WILDS] Total overlapping classes: {self.overlapping_classes}")
        print(f"[WILDS] Dataset created with {len(self.datasets)} environments")

    def metadata_values(self, wilds_dataset, metadata_name):
        metadata_index = wilds_dataset.metadata_fields.index(metadata_name)
        metadata_vals = wilds_dataset.metadata_array[:, metadata_index]
        return sorted(list(set(metadata_vals.view(-1).tolist())))


class WILDSCamelyon(WILDSDataset):
    CHECKPOINT_FREQ = 300
    ENVIRONMENTS = [
        "hospital_0",
        "hospital_1",
        "hospital_2",
        "hospital_3",
        "hospital_4",
    ]
    NUM_CLASSES = 2  # Binary: 0=normal, 1=tumor

    # Overlap configurations matching PACS/VLCS structure
    # For 4 source domains (when 1 is held out as test)
    OVERLAP_CONFIG = {
        "0": [[0], [1], [0, 1], [0]],  # Minimal overlap
        "low": [[0], [0, 1], [1], [0, 1]],  # Some overlap
        "high": [[0, 1], [0, 1], [0, 1], [0, 1]],  # Full overlap
        "100": [[0, 1], [0, 1], [0, 1], [0, 1]],  # All classes everywhere
        "low_linked_only": [[0], [], [1], []],  # Sparse
        "high_linked_only": [[0], [], [], []],  # Very sparse
    }

    def __init__(
        self,
        root: str,
        test_envs: list,
        hparams: dict,
        overlap_type: str,
        overlap_seed=None,
        num_classes=None,
        num_domain_linked_classes=None
    ):
        self.dir = os.path.join(root, "camelyon17_v1.0/")
        self._num_source_domains = 4  # 5 hospitals - 1 test = 4 source
        self._num_classes = 2

        print(f"[WILDSCamelyon] Initializing with overlap_type: {overlap_type}")
        print(f"[WILDSCamelyon] Test environments: {test_envs}")

        dataset = Camelyon17Dataset(root_dir=root)

        print(f"[WILDSCamelyon] Loaded base dataset: {len(dataset)} total samples")
        print(f"[WILDSCamelyon] Number of classes: {dataset.n_classes}")

        super().__init__(
            dataset,
            "hospital",
            test_envs,
            hparams["data_augmentation"],
            hparams,
            WILDSCamelyon.OVERLAP_CONFIG[overlap_type],
        )


# if __name__ == '__main__':
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--data_dir", type=str, default="./data")
#     parser.add_argument("--overlap", type=str, default="high")
#     args = parser.parse_args()

#     dataset = WILDSCamelyon(
#         root=args.data_dir,
#         test_envs=[0],
#         hparams={'data_augmentation': True},
#         overlap_type=args.overlap
#     )

#     print(f"\n=== Dataset Summary ===")
#     print(f"Number of classes: {dataset.num_classes}")
#     print(f"Overlapping classes: {dataset.overlapping_classes}")
#     print(f"Number of environments: {len(dataset.datasets)}")

#     for i, env in enumerate(dataset.datasets):
#         print(f"  {env.name}: {len(env)} samples (test={env.is_test_env})")

## INIT

In [ ]:
DATASETS = {
    "PACS": PACS,
    "VLCS": VLCS,
    "OfficeHome": OfficeHome,
    #"WILDSCamelyon": WILDSCamelyon
}


# Networks

## Base


# MixStyle

In [ ]:
# -----------------------------
# Networks Base (fixed: MixStyle detection)
# -----------------------------
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

# -----------------------------
# MixStyle module
# -----------------------------
class MixStyle(nn.Module):
    def __init__(self, p: float = 0.5, alpha: float = 0.1, eps: float = 1e-6):
        super(MixStyle, self).__init__()
        self.p = float(p)
        self.alpha = float(alpha)
        self.eps = float(eps)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if (not self.training) or (torch.rand(1).item() > self.p):
            return x
        B, C, H, W = x.size()
        mu = x.mean(dim=[2, 3], keepdim=True)
        sigma = x.std(dim=[2, 3], keepdim=True).clamp(min=self.eps)
        idx = torch.randperm(B, device=x.device)
        mu2 = mu[idx]
        sigma2 = sigma[idx]
        lam = torch.distributions.Beta(self.alpha, self.alpha).sample().to(x.device) if self.alpha > 0 else 0.5
        x_normed = (x - mu) / sigma
        mixed_sigma = lam * sigma + (1 - lam) * sigma2
        mixed_mu = lam * mu + (1 - lam) * mu2
        x_mixed = x_normed * mixed_sigma + mixed_mu
        return x_mixed


# -----------------------------
# Identity layer
# -----------------------------
class Identity(nn.Module):
    def forward(self, x):
        return x


# -----------------------------
# ResNet with MixStyle and train()
# -----------------------------
class ResNet(nn.Module):
    def __init__(self, input_shape, hparams):
        super(ResNet, self).__init__()
        resnet18_flag = hparams.get("resnet18", True)
        self.hparams = hparams

        # Load backbone
        if resnet18_flag:
            self._load_resnet_variant(torchvision.models.resnet18,
                                      torchvision.models.ResNet18_Weights.IMAGENET1K_V1,
                                      pretrained_path="~/scratch/saved/resnet18-f37072fd.pth")
            self.n_outputs = 512
        else:
            self._load_resnet_variant(torchvision.models.resnet50,
                                      torchvision.models.ResNet50_Weights.IMAGENET1K_V2,
                                      pretrained_path="~/scratch/saved/resnet50-11ad3fa6.pth")
            self.n_outputs = 2048

        # Adapt first conv if input channels != 3
        nc = input_shape[0]
        if nc != 3:
            tmp = self.network.conv1.weight.data.clone()
            self.network.conv1 = nn.Conv2d(nc, 64, kernel_size=7, stride=2, padding=3, bias=False)
            for i in range(nc):
                self.network.conv1.weight.data[:, i, :, :] = tmp[:, i % 3, :, :]

        # Remove final fc
        self.network.fc = Identity()

        # MixStyle module
        ms_p = hparams.get("mixstyle_p", 0.5)
        ms_alpha = hparams.get("mixstyle_alpha", 0.1)
        self.mixstyle = MixStyle(p=ms_p, alpha=ms_alpha)
        self.use_mixstyle = True  # <- required for sanity check

        # Dropout
        self.dropout = nn.Dropout(hparams.get("resnet_dropout", 0.0))

        # Freeze BN
        self.freeze_bn()

    def _load_resnet_variant(self, constructor_fn, weights_enum, pretrained_path=None):
        pretrained_path = os.path.expanduser(pretrained_path) if pretrained_path else None
        if pretrained_path and os.path.exists(pretrained_path):
            print(f"[info] loading weights from {pretrained_path}")
            model = constructor_fn()
            state = torch.load(pretrained_path, map_location="cpu")
            model.load_state_dict(state)
            self.network = model
        else:
            self.network = constructor_fn(weights=weights_enum)

    def forward(self, x):
        x = self.network.conv1(x)
        x = self.network.bn1(x)
        x = self.network.relu(x)
        x = self.network.maxpool(x)
        x = self.network.layer1(x)
        # Debug print to confirm MixStyle activation
        if hasattr(self, "mixstyle") and self.training:
            # print(">> MixStyle ACTIVE on this batch")
            x = self.mixstyle(x)
        x = self.network.layer2(x)
        x = self.network.layer3(x)
        x = self.network.layer4(x)
        x = self.network.avgpool(x)
        x = torch.flatten(x, 1)
        return self.dropout(x)

    def train(self, mode: bool = True):
        super(ResNet, self).train(mode)
        self.freeze_bn()
        return self

    def freeze_bn(self):
        for m in self.network.modules():
            if isinstance(m, nn.BatchNorm2d):
                m.eval()
                if hasattr(m, "weight") and m.weight is not None:
                    m.weight.requires_grad = False
                if hasattr(m, "bias") and m.bias is not None:
                    m.bias.requires_grad = False


# -----------------------------
# Featurizer / Classifier / Algorithm / ERM
# -----------------------------
def Featurizer(input_shape, hparams):
    if input_shape[1:3] == (224, 224):
        return ResNet(input_shape, hparams)
    else:
        raise NotImplementedError("Featurizer only supports 224x224 inputs.")


def Classifier(in_features, out_features, is_nonlinear=False):
    if is_nonlinear:
        return nn.Sequential(
            nn.Linear(in_features, in_features // 2),
            nn.ReLU(),
            nn.Linear(in_features // 2, in_features // 4),
            nn.ReLU(),
            nn.Linear(in_features // 4, out_features),
        )
    else:
        return nn.Linear(in_features, out_features)


class Algorithm(nn.Module):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(Algorithm, self).__init__()
        self.hparams = hparams

    def update(self, minibatches, unlabeled=None):
        raise NotImplementedError

    def predict(self, x):
        raise NotImplementedError


class ERM(Algorithm):
    use_mixstyle = True  # <- required for sanity check

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(ERM, self).__init__(input_shape, num_classes, num_domains, hparams)
        self.featurizer = Featurizer(input_shape, self.hparams)
        self.classifier = Classifier(
            self.featurizer.n_outputs, num_classes, self.hparams.get("nonlinear_classifier", False)
        )
        self.network = nn.Sequential(self.featurizer, self.classifier)
        self.optimizer = torch.optim.Adam(
            self.network.parameters(),
            lr=self.hparams.get("lr", 1e-4),
            weight_decay=self.hparams.get("weight_decay", 0.0),
        )

    def train(self, mode: bool = True):
        super(ERM, self).train(mode)
        self.featurizer.train(mode)  # ensures MixStyle is active
        return self

    def update(self, minibatches, unlabeled=None):
        all_x = torch.cat([x for x, y in minibatches])
        all_y = torch.cat([y for x, y in minibatches])
        loss = F.cross_entropy(self.predict(all_x), all_y)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        return {"loss": loss.item()}

    def predict(self, x):
        return self.network(x)


## Fond

In [ ]:


class AbstractXDom(ERM):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(AbstractXDom, self).__init__(
            input_shape, num_classes, num_domains, hparams
        )

        self.domain_relations = hparams.get("domain_relations", None)

        self.temperature = hparams["temperature"]
        self.base_temperature = hparams["base_temperature"]

        encoder_output = 512 if hparams["resnet18"] else 2048
        self.projector = nn.Sequential(
            nn.Linear(encoder_output, encoder_output),
            nn.ReLU(),
            nn.Linear(encoder_output, 256),
        )

        def weight_init(m):
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        self.projector.apply(weight_init)

        self.optimizer = torch.optim.Adam(
            (
                list(self.featurizer.parameters())
                + list(self.classifier.parameters())
                + list(self.projector.parameters())
            ),
            lr=self.hparams["lr"],
            weight_decay=self.hparams["weight_decay"],
        )

    def get_masks(self, Y, D):
        """
        Generate masks relating samples and their domains/classes
        """
        # mask-out self-contrast cases
        self_mask = (~torch.eye(Y.shape[0], dtype=torch.bool)).to(Y.device)

        # mask out dot products between different classes
        same_Y_mask = torch.eq(Y.view(-1, 1), Y.view(-1, 1).T)
        # mask out dot products between different domains
        same_D_mask = torch.eq(D.view(-1, 1), D.view(-1, 1).T)

        return {
            "self_mask": self_mask,
            "same_class_mask": same_Y_mask,
            "same_class_exclude_self_mask": same_Y_mask * self_mask,
            "same_domain_mask": same_D_mask,
            "diff_domain_mask": ~same_D_mask,
            "diff_domain_same_class_mask": same_Y_mask * ~same_D_mask,
            "same_domain_diff_class_mask": ~same_Y_mask * same_D_mask,
            "same_domain_same_class_mask": same_Y_mask * same_D_mask,
        }

    def supcon_loss(
        self,
        projections,
        positive_mask,
        negative_mask,
        alpha: torch.Tensor,
        beta: torch.Tensor,
        epsilon: float = 1e-6,
    ):
        """
        Regular FOND_FBA Loss with custom masks for positive and A(i) "negative" samples
        """

        mean_positives_per_sample = (
            torch.count_nonzero(positive_mask) / projections.shape[0]
        )

        # count the number of samples with no positives
        num_zero_positives = projections.shape[0] - torch.count_nonzero(
            positive_mask.sum(1)
        )

        # proj_dot is cos similarity b/c features are normalized
        # find the dot product with respect to every x
        proj_dot = torch.div(torch.matmul(projections, projections.T), self.temperature)

        # for numeric stability so sum is never zero
        logits_max, _ = torch.max(proj_dot, dim=1, keepdim=True)
        logits = proj_dot - logits_max.detach()

        # compute exp per element (i.e. over each cosine similarity)
        exp_logits = torch.exp(logits) * negative_mask * beta
        # weigh intra domain negatives higher = same domain different class

        # decompose log(exp(x)/y) = x - log(y)
        # y = summation of cos similarities excluding self
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positives
        mean_log_prob_pos = (positive_mask * alpha * log_prob).sum(1) / (
            positive_mask.sum(1) + epsilon
        )

        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.mean()

        return loss, mean_positives_per_sample, num_zero_positives

    def preprocess(self, minibatches):
        # NOTE: current implementations doesn't create duplicates
        # check SelfReg

        num_domains = len(minibatches)
        features = [self.featurizer(xi) for xi, _ in minibatches]

        projections = [F.normalize(self.projector(fi)) for fi in features]
        classifs = [self.classifier(fi) for fi in features]
        targets = [yi for _, yi in minibatches]

        # create domain labels
        domains = [
            torch.zeros(len(x), dtype=torch.uint8).to(x.device) + i
            for i, x in enumerate(targets)
        ]

        # match domains
        if self.domain_relations is not None:
            for old, new in self.domain_relations.items():
                for d in domains:
                    d[d == old] = new

        return {
            "features": features,
            "projections": projections,
            "classifs": classifs,
            "targets": targets,
            "domains": domains,
            "num_domains": num_domains,
        }

    def update(self, minibatches, unlabeled=None):
        raise NotImplementedError()


class FONDBase(AbstractXDom):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FONDBase, self).__init__(input_shape, num_classes, num_domains, hparams)

        # hparams
        self.xdom_lmbd = hparams["xdom_lmbd"]
        self.error_lmbd = hparams["error_lmbd"]
        self.xda_alpha = hparams["xda_alpha"]
        self.xda_beta = hparams["xda_beta"]
        self.C_oc = hparams["C_oc"]

        # create class masks
        oc_weight = torch.zeros(num_classes, dtype=torch.bool)
        oc_weight[self.C_oc] = True
        noc_weight = ~oc_weight
        self.oc_weight = oc_weight.type(torch.float)
        self.noc_weight = noc_weight.type(torch.float)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)

        class_loss = F.cross_entropy(classifs, targets)

        loss = (
            class_loss
            + self.xdom_lmbd * xdom_loss
            + self.error_lmbd * torch.abs(error_loss)
        )

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


class FOND(FONDBase):
    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND, self).__init__(input_shape, num_classes, num_domains, hparams)


class FOND_NC(FONDBase):
    """
    Based on FOND however we replace the fairness loss with the domain-linked
    classification loss
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND_NC, self).__init__(input_shape, num_classes, num_domains, hparams)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)
        class_loss = F.cross_entropy(classifs, targets)
        if torch.isnan(noc_class_loss):
            noc_class_loss = torch.tensor(0).to(targets.device)

        loss = (
            class_loss + self.xdom_lmbd * xdom_loss + self.error_lmbd * noc_class_loss
        )

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "noc_class_loss": noc_class_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


# FOND with a non overlapping class loss, but no overall class loss
class FOND_N(FONDBase):
    """
    Guiding Question: Why optimize for domain-shared class accuracy if
    we are only interested in domain-linked classes? Does including domain-shared
    classes for the contrastive objective only improve domain-linked class
    performance?

    Based on FOND however we keep the domain-shared and domain-linked
    contrastive loss and only optimize for the domain-linked classification
    loss.
    """

    def __init__(self, input_shape, num_classes, num_domains, hparams):
        super(FOND_N, self).__init__(input_shape, num_classes, num_domains, hparams)

    def update(self, minibatches, unlabeled=None):
        values = self.preprocess(minibatches)

        domains = torch.cat(values["domains"])
        targets = torch.cat(values["targets"])
        classifs = torch.cat(values["classifs"])
        projections = torch.cat(values["projections"])

        masks = self.get_masks(Y=targets, D=domains)

        alpha = (
            ~masks["diff_domain_same_class_mask"]
            + masks["diff_domain_same_class_mask"] * self.xda_alpha
        )

        beta = (
            ~masks["same_domain_diff_class_mask"]
            + masks["same_domain_diff_class_mask"] * self.xda_beta
        )

        xdom_loss, mean_positives_per_sample, num_zero_positives = self.supcon_loss(
            projections=projections,
            positive_mask=masks["same_class_mask"] * masks["self_mask"],
            negative_mask=masks["self_mask"],
            alpha=alpha,
            beta=beta,
        )

        oc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.oc_weight.to(targets.device)
        )
        noc_class_loss = F.cross_entropy(
            classifs, targets, weight=self.noc_weight.to(targets.device)
        )
        error_loss = oc_class_loss - noc_class_loss
        if torch.isnan(error_loss):
            error_loss = torch.tensor(0).to(targets.device)
        class_loss = F.cross_entropy(classifs, targets)
        if torch.isnan(noc_class_loss):
            noc_class_loss = torch.tensor(0).to(targets.device)

        loss = self.xdom_lmbd * xdom_loss + noc_class_loss

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        return {
            "loss": loss.item(),
            "class_loss": class_loss.item(),
            "xdom_loss": xdom_loss.item(),
            "error_loss": error_loss.item(),
            "noc_class_loss": noc_class_loss.item(),
            "mean_p": mean_positives_per_sample.item(),
            "zero_p": num_zero_positives.item(),
        }


# DomainBalance Mini Batch

In [ ]:
# --- DOMAIN-BALANCED MINI-BATCH SETUP ---
from torch.utils.data import DataLoader, ConcatDataset, Sampler
import torch
import math

class DomainBalancedSampler(Sampler):
    """
    Samples batches ensuring each source domain contributes equally.
    """
    def __init__(self, domain_datasets, batch_size):
        self.domain_datasets = domain_datasets
        self.batch_size = batch_size
        self.num_domains = len(domain_datasets)
        self.domain_lengths = [len(d) for d in domain_datasets]

        self.per_domain_batch = batch_size // self.num_domains
        assert self.per_domain_batch > 0, "Batch size too small for number of domains"

        self.domain_indices = [torch.randperm(l).tolist() for l in self.domain_lengths]
        self.num_batches = max([math.ceil(l / self.per_domain_batch) for l in self.domain_lengths])

    def __iter__(self):
        for batch_idx in range(self.num_batches):
            batch = []
            for d, indices in enumerate(self.domain_indices):
                start = batch_idx * self.per_domain_batch
                end = start + self.per_domain_batch
                domain_batch = indices[start:end]
                if len(domain_batch) < self.per_domain_batch:
                    reshuffled = torch.randperm(len(self.domain_datasets[d])).tolist()
                    domain_batch += reshuffled[: self.per_domain_batch - len(domain_batch)]
                    self.domain_indices[d] = reshuffled
                batch.extend(domain_batch)
            yield batch

    def __len__(self):
        return self.num_batches

# --- Dynamically get dataset object ---
# This assumes `dataset_name` will be assigned later (e.g., "OfficeHome")
# The `dataset_obj` variable will be used in training code
dataset_obj = None  # placeholder, will be assigned after dataset is initialized

def create_domain_balanced_loader(dataset_obj, batch_size=32, num_workers=4):
    """
    Returns a DataLoader that balances samples across source domains
    """
    source_env_datasets = [
        dataset_obj.datasets[i] for i in range(len(dataset_obj.datasets))
        if i not in dataset_obj.test_envs
    ]
    sampler = DomainBalancedSampler(source_env_datasets, batch_size)
    combined_dataset = ConcatDataset(source_env_datasets)
    balanced_loader = DataLoader(
        combined_dataset,
        batch_sampler=sampler,
        num_workers=num_workers,
        pin_memory=True,
    )
    print(f"[INFO] Domain-balanced loader created with {len(source_env_datasets)} source domains.")
    print(f"[INFO] Total samples in loader: {len(combined_dataset)}")
    return balanced_loader

# Example usage after dataset object is initialized:
# dataset_obj = OfficeHome(root="data/OfficeHome", test_envs=[0], hparams=hparams, overlap_type="high")
# balanced_loader = create_domain_balanced_loader(dataset_obj, batch_size=32)


## Init

In [ ]:


ALGORITHMS = {
    "ERM": ERM,
    "FOND": FOND,
    # "FOND_NC": "FOND_NC",
    # "FOND_N": "FOND_N",
    # "FOND_Distillation_Separate_Projector": "FOND_Distillation_Separate_Projector",
    # "FOND_Distillation_Teacher_Projector": "FOND_Distillation_Teacher_Projector",
    # "FOND_Distillation_Student_Projector": "FOND_Distillation_Student_Projector",
}


# DOmain Mini Batch Setup

In [ ]:
# ------------------------------
# Domain-Balanced Mini-batch Setup
# ------------------------------
from torch.utils.data import DataLoader, ConcatDataset

# Ensure the dataset object exists before using it
try:
    dataset_obj
except NameError:
    dataset_obj = None
    print("Dataset object not yet defined. DomainBalancedSampler will be initialized after dataset creation.")

if dataset_obj is not None:
    # Extract source domains (exclude test_envs)
    source_env_datasets = [dataset_obj[i] for i in range(len(dataset_obj)) if i not in config["test_set_id"]]

    # Use batch_size from config if exists, else default
    batch_size = config.get("batch_size", 32)

    # Initialize the sampler
    sampler = DomainBalancedSampler(source_env_datasets, batch_size)

    # Concatenate all source envs
    combined_dataset = ConcatDataset(source_env_datasets)

    # DataLoader with domain-balanced sampling
    balanced_loader = DataLoader(
        combined_dataset,
        batch_sampler=sampler,  # <-- Corrected: use batch_sampler, not sampler
        num_workers=config.get("num_workers", 4),
        pin_memory=True,
    )

    print(f"[INFO] Domain-balanced loader initialized with {len(source_env_datasets)} source environments.")
    print(f"[INFO] Total samples in loader: {len(combined_dataset)}")
else:
    print("Skipping DomainBalancedSampler initialization since dataset is not defined yet.")


# Train

# SWAD

## Fit Simple

In [ ]:
def fit_simple(
    exp_dir: str,
    logger,  # CSVLogger or PrintLogger
    seed: int,
    trial_seed: int,
    hparams_seed: int,
    algorithm_name: str,
    dataset_name: str,
    data_dir: str,
    num_workers: int,
    test_envs: list,
    overlap_type: str,
    holdout_fraction: float = 0.2,
    n_steps: int = 5001,
    checkpoint_freq: int = 300,
    model_checkpoint=None,
    teacher_paths=None,
    num_domain_linked_classes=None,
    num_classes=None,
    auto_augment: bool = False,
    augment_search_epochs: int = 10,
    use_swad: bool = True,
    swad_start_epoch: int = 1
):
    import collections, os, json, time, logging
    import numpy as np
    import torch
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ----------------------------
    # SEED AND DEVICE
    # ----------------------------
    L.seed_everything(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ----------------------------
    # HYPERPARAMETERS
    # ----------------------------
    if hparams_seed == -1:
        # === OPTUNA-TUNED HYPERPARAMETERS ===
        hparams = default_hparams(algorithm_name, dataset_name)
        hparams.update(BEST_HPARAMS)
        logging.info("Using Optuna-tuned hyperparameters")
    else:
        if hparams_seed == 0:
            hparams = default_hparams(algorithm_name, dataset_name)
        else:
            hparams = random_hparams(
                algorithm_name,
                dataset_name,
                seed_hash(hparams_seed, trial_seed)
            )

    logging.info(f"hparams: {hparams}")

    # ----------------------------
    # LOAD DATASET
    # ----------------------------
    dataset = DATASETS[dataset_name](
        root=data_dir,
        test_envs=test_envs,
        hparams=hparams,
        overlap_type=overlap_type,
        num_classes=num_classes,
        num_domain_linked_classes=num_domain_linked_classes,
    )
    hparams["C_oc"] = dataset.overlapping_classes
    logging.info(f"Loaded {dataset_name}")

    # ----------------------------
    # SPLIT ENVIRONMENTS
    # ----------------------------
    in_splits, out_splits = [], []
    relative_test_env = None
    log_dir = logger.return_root()

    for env_i, env in enumerate(dataset):
        out, in_ = split_dataset(
            env,
            int(len(env) * holdout_fraction),
            seed_hash(trial_seed, env_i)
        )

        if hparams.get("class_balanced", False):
            in_weights = make_weights_for_balanced_classes(in_)
            out_weights = make_weights_for_balanced_classes(out)
        else:
            in_weights, out_weights = None, None

        in_splits.append((in_, in_weights))
        out_splits.append((out, out_weights))

        if env.is_test_env:
            relative_test_env = env_i

    assert relative_test_env is not None, "No testing domains"
    logging.info(f"test_envs={test_envs}, relative_test_env={relative_test_env}")

    # ----------------------------
    # DATA LOADERS
    # ----------------------------
    train_loaders = [
        InfiniteDataLoader(
            dataset=env,
            weights=env_weights,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for i, (env, env_weights) in enumerate(in_splits)
        if i != relative_test_env
    ]

    eval_loaders = [
        FastDataLoader(
            dataset=env,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for env, _ in (in_splits + out_splits)
    ]

    eval_weights = [None for _ in (in_splits + out_splits)]

    eval_loader_names = (
        [f"env{i}_in" for i in range(len(in_splits))] +
        [f"env{i}_out" for i in range(len(out_splits))]
    )

    logging.info(f"Created data loaders: {eval_loader_names}")

    train_minibatches_iterator = zip(*train_loaders)
    steps_per_epoch = min(
        [len(env) / hparams["batch_size"] for env, _ in in_splits]
    )

    # ----------------------------
    # INITIALIZE ALGORITHM
    # ----------------------------
    algorithm = ALGORITHMS[algorithm_name](
        input_shape=dataset.input_shape,
        num_classes=dataset.num_classes,
        num_domains=len(dataset) - len(test_envs),
        hparams=hparams,
    )
    algorithm.to(device)
    logging.info(f"Algorithm {algorithm_name} initialized")

    # ----------------------------
    # MODEL STATISTICS
    # ----------------------------
    total_params = sum(p.numel() for p in algorithm.parameters())
    trainable_params = sum(p.numel() for p in algorithm.parameters() if p.requires_grad)
    model_size_mb = sum(
        p.nelement() * p.element_size() for p in algorithm.parameters()
    ) / (1024 ** 2)

    logging.info(f"Total params: {total_params:,}")
    logging.info(f"Trainable params: {trainable_params:,}")
    logging.info(f"Model size: {model_size_mb:.2f} MB")

    # ----------------------------
    # SWAD INITIALIZATION
    # ----------------------------
    if use_swad:
        swad_weights = {
            name: p.clone().detach()
            for name, p in algorithm.named_parameters()
            if p.requires_grad
        }
        swad_n = 0

    # ----------------------------
    # TRAINING LOOP
    # ----------------------------
    checkpoint_vals = collections.defaultdict(list)
    training_start_time = time.time()
    epoch_start_time = time.time()
    current_epoch = 0
    steps_in_current_epoch = 0

    for step in tqdm(range(n_steps)):
        minibatches_device = [
            (x.to(device), y.to(device))
            for x, y in next(train_minibatches_iterator)
        ]

        step_vals = algorithm.update(minibatches_device, None)
        for k, v in step_vals.items():
            checkpoint_vals[k].append(v)

        # ---- SWAD UPDATE ----
        if use_swad and current_epoch >= swad_start_epoch:
            swad_n += 1
            for name, p in algorithm.named_parameters():
                if p.requires_grad:
                    swad_weights[name] = (
                        (swad_n - 1) * swad_weights[name] + p.detach()
                    ) / swad_n

        # ---- Epoch tracking ----
        steps_in_current_epoch += 1
        if steps_in_current_epoch >= steps_per_epoch:
            current_epoch += 1
            steps_in_current_epoch = 0
            epoch_start_time = time.time()

        # ---- Evaluation ----
        if step % checkpoint_freq == 0 or step == n_steps - 1:
            if use_swad and swad_n > 0:
                original_params = {
                    n: p.clone() for n, p in algorithm.named_parameters()
                }
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(swad_weights[n])

            results_dict = {
                key: {
                    _key: []
                    for _key in ["acc", "f1", "nacc", "oacc", "recall", "precision", "loss"]
                }
                for key in ["train", "val", "test", "other"]
            }

            # Calculate training value averages
            for key, val in checkpoint_vals.items():
                results_dict["train"][str(key)] = [np.mean(val)]

            # Evaluation
            for name, loader, weights in zip(
                eval_loader_names, eval_loaders, eval_weights
            ):
                (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
                    algorithm, loader, weights, device, dataset
                )
                loss = compute_loss(algorithm, loader, device)

                domain_idx = int(name[3])

                if domain_idx == relative_test_env:
                    if "in" in name:
                        loader_type = "test"
                    else:
                        loader_type = "other"
                elif "out" in name:
                    loader_type = "val"
                else:
                    loader_type = "train"

                results_dict[loader_type]["acc"].append(float(acc))
                results_dict[loader_type]["recall"].append(float(recall))
                results_dict[loader_type]["f1"].append(float(f1))
                results_dict[loader_type]["precision"].append(float(precision))
                results_dict[loader_type]["nacc"].append(float(nacc))
                results_dict[loader_type]["oacc"].append(float(oacc))
                results_dict[loader_type]["loss"].append(float(loss))

            # Log metrics
            current_epoch_num = step / steps_per_epoch
            print(f"\n=== Step {step} (Epoch {current_epoch_num:.2f}) ===")

            for stage in ["train", "val", "test", "other"]:
                if results_dict[stage]["acc"]:
                    print(f"{stage.upper()}:")
                    print(f"  loss: {np.mean(results_dict[stage]['loss']):.4f}")
                    print(f"  acc: {np.mean(results_dict[stage]['acc']):.4f}")
                    print(f"  precision: {np.mean(results_dict[stage]['precision']):.4f}")
                    print(f"  recall: {np.mean(results_dict[stage]['recall']):.4f}")
                    print(f"  f1: {np.mean(results_dict[stage]['f1']):.4f}")
                    print(f"  oacc: {np.mean(results_dict[stage]['oacc']):.4f}")
                    print(f"  nacc: {np.mean(results_dict[stage]['nacc']):.4f}")

            logger.log(results_dict, step)
            checkpoint_vals.clear()

            if use_swad and swad_n > 0:
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(original_params[n])

    # ----------------------------
    # TRAINING COMPLETION SUMMARY
    # ----------------------------
    total_training_time = time.time() - training_start_time
    logging.info("=" * 80)
    logging.info("TRAINING COMPLETED")
    logging.info("=" * 80)
    logging.info(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    logging.info(f"Total steps: {n_steps}")
    logging.info(f"Total epochs: {current_epoch}")
    logging.info("=" * 80)

    print("Training completed.")

# Train AutoFOND

## Setup

In [ ]:
import os
from pathlib import Path
import json
import logging

# ----------------------------
# Setup output directories
# ----------------------------
output_dir = "/kaggle/working/experiments"
os.makedirs(output_dir, exist_ok=True)

# Get experiment name from input
exp_name_oh = input("Enter experiment name: ").strip()
config["name"] = exp_name_oh

# If name not provided, generate default
exp_name_oh = config["name"] or f"{config['algo']}_{config['dataset']}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
exp_dir = Path(output_dir) / exp_name_oh
exp_dir.mkdir(parents=True, exist_ok=True)

log_dir = exp_dir / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Setup logging
# ----------------------------
config_logging()  # your function to setup logging
logging.info(f"Starting experiment: {exp_name_oh}")
logging.info(f"Output directory: {exp_dir}")

# ----------------------------
# Save config
# ----------------------------
with open(exp_dir / "config.json", "w") as f:
    json.dump(config, f, indent=2)

# ----------------------------
# Initialize CSV logger
# ----------------------------
logger = CSVLogger(log_dir / 'metrics.csv', root_dir=log_dir)


## Hyperparameter Tuning of OfficeHome using Optuna

In [ ]:
dataset_name = config["dataset"][2]
dataset_name

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
import optuna
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Seeds
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader
from torchvision import transforms

DATA_DIR = "/kaggle/input/officehome/OfficeHome"

# Standard transform for ResNet input
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def load_officehome_dataset(split="train", val_ratio=0.2, seed=0):
    """
    Load OfficeHome dataset and split into train/val.
    """
    full_dataset = ImageFolder(root=DATA_DIR, transform=base_transform)

    total_len = len(full_dataset)
    val_len = int(total_len * val_ratio)
    train_len = total_len - val_len

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len], generator=generator)

    if split == "train":
        return train_dataset
    else:
        return val_dataset


In [ ]:
def objective(trial):
    # --- Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    scheduler_name = trial.suggest_categorical("scheduler", ["StepLR", "CosineAnnealingLR"])

    # --- Load datasets ---
    train_dataset = load_officehome_dataset(split="train")
    val_dataset   = load_officehome_dataset(split="val")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # --- Model ---
    model = models.resnet18(pretrained=True)
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, 65)
    )
    model = model.to(device)

    # --- Optimizer ---
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    # --- Scheduler ---
    if scheduler_name == "StepLR":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    # --- Loss ---
    criterion = nn.CrossEntropyLoss()

    # --- Training loop (small for tuning) ---
    best_val_acc = 0
    for epoch in range(3):  # small number of epochs for tuning
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                y_hat = model(x)
                correct += (y_hat.argmax(1) == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return best_val_acc


In [ ]:
# --- Create Optuna study and run hyperparameter optimization ---
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)  # adjust n_trials if you want more tuning

# --- Print best hyperparameters ---
best = study.best_params
print("Best Hyperparameters:", best)


In [ ]:
# --- Save best hyperparameters from Optuna ---
best = study.best_params
print("Best Hyperparameters:", best)

# Example output format:
# {'lr': 0.000623, 'batch_size': 64, 'optimizer': 'SGD',
#  'weight_decay': 0.00668, 'aug_strength': 0.0713,
#  'scheduler': 'CosineAnnealingLR', 'dropout': 0.118}


In [ ]:
# --- Load datasets using best hyperparameters ---
train_dataset = load_officehome_dataset(split="train")
val_dataset   = load_officehome_dataset(split="val")
test_dataset  = val_dataset  # Use separate test set if available

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=best["batch_size"], shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=best["batch_size"], shuffle=False, num_workers=4)


In [ ]:
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(best["dropout"]),
    nn.Linear(model.fc.in_features, 65)
)
model = model.to(device)

if best["optimizer"] == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=best["lr"], momentum=0.9, weight_decay=best["weight_decay"])

if best["scheduler"] == "StepLR":
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

criterion = nn.CrossEntropyLoss()


In [ ]:
train_acc_list, val_acc_list, test_acc_list = [], [], []
train_loss_list, val_loss_list, test_loss_list = [], [], []

num_epochs = 10
for epoch in range(num_epochs):
    # ===== TRAIN =====
    model.train()
    running_loss, correct, total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_hat = model(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (y_hat.argmax(1)==y).sum().item()
        total += y.size(0)
    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)
    scheduler.step()

    # ===== VALIDATION =====
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(1)==y).sum().item()
            total += y.size(0)
    val_loss = running_loss / total
    val_acc = correct / total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    # ===== TEST =====
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(1)==y).sum().item()
            total += y.size(0)
    test_loss = running_loss / total
    test_acc = correct / total
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# ---- Loss Plot ----
plt.figure(figsize=(10,5))
plt.plot(train_loss_list, label="Train Loss", marker='o')
plt.plot(val_loss_list, label="Validation Loss", marker='s')
plt.plot(test_loss_list, label="Test Loss", marker='^')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train / Validation / Test Loss")
plt.legend()
plt.grid(True)
plt.show()

# ---- Accuracy Plot ----
plt.figure(figsize=(10,5))
plt.plot(train_acc_list, label="Train Accuracy", marker='o')
plt.plot(val_acc_list, label="Validation Accuracy", marker='s')
plt.plot(test_acc_list, label="Test Accuracy", marker='^')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train / Validation / Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Get all predictions and true labels
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        preds = y_hat.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# Classification Report
print("Classification Report:")
print(classification_report(all_labels, all_preds, digits=4))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

# Class names for OfficeHome
class_names = ["Art", "Clipart", "Product", "RealWorld"]

plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()



# Training Officehome using Best hyperparameters

In [ ]:
# =========================================================
# Inject Optuna best hyperparameters into fit_simple
# =========================================================

BEST_HPARAMS = {
    "lr": 8.969255213114905e-05,
    "batch_size": 64,
    "weight_decay": 0.0017349923812696934,
    "optimizer": "Adam",
    "resnet_dropout": 0.3774988325895403,
    "lr_scheduler": "StepLR"
}

print("Using Optuna-tuned hyperparameters:")
for k, v in BEST_HPARAMS.items():
    print(f"  {k}: {v}")


In [ ]:
def fit_simple(
    exp_dir: str,
    logger,  # CSVLogger or PrintLogger
    seed: int,
    trial_seed: int,
    hparams_seed: int,
    algorithm_name: str,
    dataset_name: str,
    data_dir: str,
    num_workers: int,
    test_envs: list,
    overlap_type: str,
    holdout_fraction: float = 0.2,
    n_steps: int = 5001,
    checkpoint_freq: int = 300,
    model_checkpoint=None,
    teacher_paths=None,
    num_domain_linked_classes=None,
    num_classes=None,
    auto_augment: bool = False,
    augment_search_epochs: int = 10,
    use_swad: bool = True,
    swad_start_epoch: int = 1
):
    import collections, os, json, time, logging
    import numpy as np
    import torch
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ----------------------------
    # SEED AND DEVICE
    # ----------------------------
    L.seed_everything(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ----------------------------
    # HYPERPARAMETERS
    # ----------------------------
    if hparams_seed == -1:
        # === OPTUNA-TUNED HYPERPARAMETERS ===
        hparams = default_hparams(algorithm_name, dataset_name)
        hparams.update(BEST_HPARAMS)
        logging.info("Using Optuna-tuned hyperparameters")
    else:
        if hparams_seed == 0:
            hparams = default_hparams(algorithm_name, dataset_name)
        else:
            hparams = random_hparams(
                algorithm_name,
                dataset_name,
                seed_hash(hparams_seed, trial_seed)
            )

    logging.info(f"hparams: {hparams}")

    # ----------------------------
    # LOAD DATASET
    # ----------------------------
    dataset = DATASETS[dataset_name](
        root=data_dir,
        test_envs=test_envs,
        hparams=hparams,
        overlap_type=overlap_type,
        num_classes=num_classes,
        num_domain_linked_classes=num_domain_linked_classes,
    )
    hparams["C_oc"] = dataset.overlapping_classes
    logging.info(f"Loaded {dataset_name}")

    # ----------------------------
    # SPLIT ENVIRONMENTS
    # ----------------------------
    in_splits, out_splits = [], []
    relative_test_env = None
    log_dir = logger.return_root()

    for env_i, env in enumerate(dataset):
        out, in_ = split_dataset(
            env,
            int(len(env) * holdout_fraction),
            seed_hash(trial_seed, env_i)
        )

        if hparams.get("class_balanced", False):
            in_weights = make_weights_for_balanced_classes(in_)
            out_weights = make_weights_for_balanced_classes(out)
        else:
            in_weights, out_weights = None, None

        in_splits.append((in_, in_weights))
        out_splits.append((out, out_weights))

        if env.is_test_env:
            relative_test_env = env_i

    assert relative_test_env is not None, "No testing domains"
    logging.info(f"test_envs={test_envs}, relative_test_env={relative_test_env}")

    # ----------------------------
    # DATA LOADERS
    # ----------------------------
    train_loaders = [
        InfiniteDataLoader(
            dataset=env,
            weights=env_weights,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for i, (env, env_weights) in enumerate(in_splits)
        if i != relative_test_env
    ]

    eval_loaders = [
        FastDataLoader(
            dataset=env,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for env, _ in (in_splits + out_splits)
    ]

    eval_weights = [None for _ in (in_splits + out_splits)]

    eval_loader_names = (
        [f"env{i}_in" for i in range(len(in_splits))] +
        [f"env{i}_out" for i in range(len(out_splits))]
    )

    logging.info(f"Created data loaders: {eval_loader_names}")

    train_minibatches_iterator = zip(*train_loaders)
    steps_per_epoch = min(
        [len(env) / hparams["batch_size"] for env, _ in in_splits]
    )

    # ----------------------------
    # INITIALIZE ALGORITHM
    # ----------------------------
    algorithm = ALGORITHMS[algorithm_name](
        input_shape=dataset.input_shape,
        num_classes=dataset.num_classes,
        num_domains=len(dataset) - len(test_envs),
        hparams=hparams,
    )
    algorithm.to(device)
    logging.info(f"Algorithm {algorithm_name} initialized")

    # ----------------------------
    # MODEL STATISTICS
    # ----------------------------
    total_params = sum(p.numel() for p in algorithm.parameters())
    trainable_params = sum(p.numel() for p in algorithm.parameters() if p.requires_grad)
    model_size_mb = sum(
        p.nelement() * p.element_size() for p in algorithm.parameters()
    ) / (1024 ** 2)

    logging.info(f"Total params: {total_params:,}")
    logging.info(f"Trainable params: {trainable_params:,}")
    logging.info(f"Model size: {model_size_mb:.2f} MB")

    # ----------------------------
    # SWAD INITIALIZATION
    # ----------------------------
    if use_swad:
        swad_weights = {
            name: p.clone().detach()
            for name, p in algorithm.named_parameters()
            if p.requires_grad
        }
        swad_n = 0

    # ----------------------------
    # TRAINING LOOP
    # ----------------------------
    checkpoint_vals = collections.defaultdict(list)
    training_start_time = time.time()
    epoch_start_time = time.time()
    current_epoch = 0
    steps_in_current_epoch = 0

    for step in tqdm(range(n_steps)):
        minibatches_device = [
            (x.to(device), y.to(device))
            for x, y in next(train_minibatches_iterator)
        ]

        step_vals = algorithm.update(minibatches_device, None)
        for k, v in step_vals.items():
            checkpoint_vals[k].append(v)

        # ---- SWAD UPDATE ----
        if use_swad and current_epoch >= swad_start_epoch:
            swad_n += 1
            for name, p in algorithm.named_parameters():
                if p.requires_grad:
                    swad_weights[name] = (
                        (swad_n - 1) * swad_weights[name] + p.detach()
                    ) / swad_n

        # ---- Epoch tracking ----
        steps_in_current_epoch += 1
        if steps_in_current_epoch >= steps_per_epoch:
            current_epoch += 1
            steps_in_current_epoch = 0
            epoch_start_time = time.time()

        # ---- Evaluation ----
        if step % checkpoint_freq == 0 or step == n_steps - 1:
            if use_swad and swad_n > 0:
                original_params = {
                    n: p.clone() for n, p in algorithm.named_parameters()
                }
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(swad_weights[n])

            results_dict = {
                key: {
                    _key: []
                    for _key in ["acc", "f1", "nacc", "oacc", "recall", "precision", "loss"]
                }
                for key in ["train", "val", "test", "other"]
            }

            # Calculate training value averages
            for key, val in checkpoint_vals.items():
                results_dict["train"][str(key)] = [np.mean(val)]

            # Evaluation
            for name, loader, weights in zip(
                eval_loader_names, eval_loaders, eval_weights
            ):
                (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
                    algorithm, loader, weights, device, dataset
                )
                loss = compute_loss(algorithm, loader, device)

                domain_idx = int(name[3])

                if domain_idx == relative_test_env:
                    if "in" in name:
                        loader_type = "test"
                    else:
                        loader_type = "other"
                elif "out" in name:
                    loader_type = "val"
                else:
                    loader_type = "train"

                results_dict[loader_type]["acc"].append(float(acc))
                results_dict[loader_type]["recall"].append(float(recall))
                results_dict[loader_type]["f1"].append(float(f1))
                results_dict[loader_type]["precision"].append(float(precision))
                results_dict[loader_type]["nacc"].append(float(nacc))
                results_dict[loader_type]["oacc"].append(float(oacc))
                results_dict[loader_type]["loss"].append(float(loss))

            # Log metrics
            current_epoch_num = step / steps_per_epoch
            print(f"\n=== Step {step} (Epoch {current_epoch_num:.2f}) ===")

            for stage in ["train", "val", "test", "other"]:
                if results_dict[stage]["acc"]:
                    print(f"{stage.upper()}:")
                    print(f"  loss: {np.mean(results_dict[stage]['loss']):.4f}")
                    print(f"  acc: {np.mean(results_dict[stage]['acc']):.4f}")
                    print(f"  precision: {np.mean(results_dict[stage]['precision']):.4f}")
                    print(f"  recall: {np.mean(results_dict[stage]['recall']):.4f}")
                    print(f"  f1: {np.mean(results_dict[stage]['f1']):.4f}")
                    print(f"  oacc: {np.mean(results_dict[stage]['oacc']):.4f}")
                    print(f"  nacc: {np.mean(results_dict[stage]['nacc']):.4f}")

            logger.log(results_dict, step)
            checkpoint_vals.clear()

            if use_swad and swad_n > 0:
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(original_params[n])

    # ----------------------------
    # TRAINING COMPLETION SUMMARY
    # ----------------------------
    total_training_time = time.time() - training_start_time
    logging.info("=" * 80)
    logging.info("TRAINING COMPLETED")
    logging.info("=" * 80)
    logging.info(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    logging.info(f"Total steps: {n_steps}")
    logging.info(f"Total epochs: {current_epoch}")
    logging.info("=" * 80)

    print("Training completed.")

In [ ]:
print("BEST_HPARAMS:", BEST_HPARAMS)


In [ ]:
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        # Map env keys to train/val/test safely
        mapping = {}
        for key in results_dict.keys():
            if "in" in key:
                mapping[key] = "train"
            elif "out" in key:
                mapping[key] = "val"
            else:
                mapping[key] = "test"

        print(f"\nStep {step} metrics:")
        for k, split in mapping.items():
            vals = results_dict[k]

            # Use len() to safely check
            acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
            loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None

            if acc is not None and loss is not None:
                print(f"  {split.upper()} ({k}) | Acc: {acc:.4f} | Loss: {loss:.4f}")


In [ ]:
import os
from pathlib import Path

# ----------------------------
# Experiment directory
# ----------------------------
EXP_DIR = "/kaggle/working/officehome_experiment"
os.makedirs(EXP_DIR, exist_ok=True)

# ----------------------------
# Logger (simple print logger or CSV logger)
# ----------------------------
class PrintLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"Step {step} metrics:")
        for split in ["train", "val", "test"]:
            if results_dict[split]["acc"]:
                print(f"  {split.upper()} | Acc: {np.mean(results_dict[split]['acc']):.4f} | Loss: {np.mean(results_dict[split]['loss']):.4f}")

logger = SafeLogger()



In [ ]:
import os
from pathlib import Path
import logging
import numpy as np
import torch

# ----------------------------
# Experiment directory
# ----------------------------
OUTPUT_DIR = "/kaggle/working/experiments"
EXP_NAME = "officehome_erm_best_hparams"
EXP_DIR = Path(OUTPUT_DIR) / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = EXP_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO)
logging.info(f"Experiment directory: {EXP_DIR}")

# ----------------------------
# Safe Logger
# ----------------------------
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"\nStep {step} metrics:")
        for split in ["train", "val", "test"]:
            if split in results_dict:
                vals = results_dict[split]
                # Safe check using len()
                acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
                loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None
                if acc is not None and loss is not None:
                    print(f"  {split.upper()} | Acc: {acc:.4f} | Loss: {loss:.4f}")

logger = SafeLogger()

# ----------------------------
# Use Optuna-tuned hyperparameters
# ----------------------------
BEST_HPARAMS = {
    'lr': 8.969255213114905e-05,
    'batch_size': 64,
    'weight_decay': 0.0017349923812696934,
    'optimizer': 'Adam',
    'resnet_dropout': 0.3774988325895403,
    'lr_scheduler': 'StepLR'
}

# ----------------------------
# Correct overlap_type for OfficeHome
# ----------------------------
VALID_OVERLAP_TYPE = "high"  # must match dataset config

# ----------------------------
# Call fit_simple to train full dataset
# ----------------------------
fit_simple(
    exp_dir=EXP_DIR,
    logger=logger,
    seed=42,                   # reproducible
    trial_seed=0,
    hparams_seed=-1,           # use BEST_HPARAMS
    algorithm_name="ERM",      # your algorithm
    dataset_name="OfficeHome",
    data_dir="/kaggle/input/officehome/OfficeHome",
    num_workers=12,
    test_envs=[0],             # Art domain as test
    overlap_type=VALID_OVERLAP_TYPE,
    holdout_fraction=0.2,
    n_steps=5001,
    checkpoint_freq=300,
    use_swad=True,
    swad_start_epoch=1
)

print("\n✅ Training of full OfficeHome dataset completed successfully.")


## Hyperparameter Tuning for PACS using Optuna

In [ ]:
dataset_name_pacs = config["dataset"][0]
dataset_name_pacs

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, models, datasets
import optuna
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Seeds
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader
from torchvision import transforms

# --- PACS dataset path ---
DATA_DIR_PACS = "/kaggle/input/pacs-dataset/kfold"  # Change this to your PACS folder path

# Standard transform for ResNet input
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def load_pacs_dataset(split="train", val_ratio=0.2, seed=0):
    """
    Load PACS dataset and split into train/val.
    PACS classes: Art, Cartoon, Photo, Sketch
    """
    full_dataset = ImageFolder(root=DATA_DIR_PACS, transform=base_transform)

    total_len = len(full_dataset)
    val_len = int(total_len * val_ratio)
    train_len = total_len - val_len

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len], generator=generator)

    if split == "train":
        return train_dataset
    else:
        return val_dataset


In [ ]:
def objective_pacs(trial):
    # --- Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    scheduler_name = trial.suggest_categorical("scheduler", ["StepLR", "CosineAnnealingLR"])

    # --- Load PACS datasets ---
    train_dataset = load_pacs_dataset(split="train")
    val_dataset   = load_pacs_dataset(split="val")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # --- Model ---
    model = models.resnet18(pretrained=True)
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, 4)  # PACS has 4 classes: Art, Cartoon, Photo, Sketch
    )
    model = model.to(device)

    # --- Optimizer ---
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    # --- Scheduler ---
    if scheduler_name == "StepLR":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    # --- Loss ---
    criterion = nn.CrossEntropyLoss()

    # --- Training loop (small for tuning) ---
    best_val_acc = 0
    for epoch in range(3):  # small number of epochs for tuning
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                y_hat = model(x)
                correct += (y_hat.argmax(1) == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return best_val_acc


In [ ]:
# --- Create Optuna study and run hyperparameter optimization for PACS ---
study = optuna.create_study(direction="maximize")
study.optimize(objective_pacs, n_trials=15)  # adjust n_trials if you want more tuning

# --- Print best hyperparameters ---
best = study.best_params
print("Best Hyperparameters for PACS:", best)


In [ ]:
# --- Save best hyperparameters from Optuna (PACS) ---
best = study.best_params
print("Best Hyperparameters for PACS:", best)

# Example output format:
# {
#   'lr': 0.000623,
#   'batch_size': 64,
#   'optimizer': 'SGD',
#   'weight_decay': 0.00668,
#   'scheduler': 'CosineAnnealingLR',
#   'dropout': 0.118
# }


In [ ]:
# --- Load PACS datasets using best hyperparameters ---
train_dataset = load_pacs_dataset(split="train")
val_dataset   = load_pacs_dataset(split="val")
test_dataset  = val_dataset  # PACS has no separate test split here

train_loader = DataLoader(
    train_dataset,
    batch_size=best["batch_size"],
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=best["batch_size"],
    shuffle=False,
    num_workers=4
)


In [ ]:
# --- Model ---
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(best["dropout"]),
    nn.Linear(model.fc.in_features, 7)  # PACS has 7 classes
)
model = model.to(device)

# --- Optimizer ---
if best["optimizer"] == "Adam":
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=best["lr"],
        weight_decay=best["weight_decay"]
    )
else:
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=best["lr"],
        momentum=0.9,
        weight_decay=best["weight_decay"]
    )

# --- Scheduler ---
if best["scheduler"] == "StepLR":
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=50,
        gamma=0.5
    )
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=50
    )

# --- Loss ---
criterion = nn.CrossEntropyLoss()


In [ ]:
train_acc_list, val_acc_list, test_acc_list = [], [], []
train_loss_list, val_loss_list, test_loss_list = [], [], []

num_epochs = 10
for epoch in range(num_epochs):

    # ===== TRAIN =====
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        y_hat = model(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (y_hat.argmax(dim=1) == y).sum().item()
        total += y.size(0)

    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)

    scheduler.step()

    # ===== VALIDATION =====
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)

            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.size(0)

    val_loss = running_loss / total
    val_acc = correct / total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    # ===== TEST =====
    running_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)

            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += y.size(0)

    test_loss = running_loss / total
    test_acc = correct / total
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Test Acc: {test_acc:.4f}"
    )


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(train_loss_list) + 1)

# ===== LOSS CURVES =====
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_loss_list, marker='o', label="Train Loss")
plt.plot(epochs, val_loss_list, marker='s', label="Validation Loss")
plt.plot(epochs, test_loss_list, marker='^', label="Test Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("PACS: Train / Validation / Test Loss")
plt.legend()
plt.grid(True)
plt.show()

# ===== ACCURACY CURVES =====
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_acc_list, marker='o', label="Train Accuracy")
plt.plot(epochs, val_acc_list, marker='s', label="Validation Accuracy")
plt.plot(epochs, test_acc_list, marker='^', label="Test Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("PACS: Train / Validation / Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import numpy as np

# ===== Collect predictions =====
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        preds = y_hat.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# ===== Get class names from dataset (CRITICAL) =====
# random_split -> Subset -> original dataset
class_names = test_loader.dataset.dataset.classes
print("Class order:", class_names)

# ===== Classification Report =====
print("PACS Classification Report:")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=class_names,
        digits=4
    )
)

# ===== Confusion Matrix =====
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("PACS Confusion Matrix")
plt.tight_layout()
plt.show()


# Training with Best Parameters for PACS

In [ ]:
# =========================================================
# Inject Optuna best hyperparameters for PACS into fit_simple
# =========================================================

BEST_HPARAMS = {
    "lr": 1.3332198891320255e-05,
    "batch_size": 16,
    "weight_decay": 9.478198291385849e-05,
    "optimizer": "Adam",
    "resnet_dropout": 0.33172328324192074,
    "lr_scheduler": "StepLR"
}

print("Using Optuna-tuned hyperparameters for PACS:")
for k, v in BEST_HPARAMS.items():
    print(f"  {k}: {v}")


In [ ]:
def fit_simple(
    exp_dir: str,
    logger,  # CSVLogger or PrintLogger
    seed: int,
    trial_seed: int,
    hparams_seed: int,
    algorithm_name: str,
    dataset_name: str,
    data_dir: str,
    num_workers: int,
    test_envs: list,
    overlap_type: str,
    holdout_fraction: float = 0.2,
    n_steps: int = 5001,
    checkpoint_freq: int = 300,
    model_checkpoint=None,
    teacher_paths=None,
    num_domain_linked_classes=None,
    num_classes=None,
    auto_augment: bool = False,
    augment_search_epochs: int = 10,
    use_swad: bool = True,
    swad_start_epoch: int = 1
):
    import collections, os, json, time, logging
    import numpy as np
    import torch
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ----------------------------
    # SEED AND DEVICE
    # ----------------------------
    L.seed_everything(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ----------------------------
    # HYPERPARAMETERS
    # ----------------------------
    if hparams_seed == -1:
        # === OPTUNA-TUNED HYPERPARAMETERS ===
        hparams = default_hparams(algorithm_name, dataset_name)
        hparams.update(BEST_HPARAMS)
        logging.info("Using Optuna-tuned hyperparameters")
    else:
        if hparams_seed == 0:
            hparams = default_hparams(algorithm_name, dataset_name)
        else:
            hparams = random_hparams(
                algorithm_name,
                dataset_name,
                seed_hash(hparams_seed, trial_seed)
            )

    logging.info(f"hparams: {hparams}")

    # ----------------------------
    # LOAD DATASET
    # ----------------------------
    dataset = DATASETS[dataset_name](
        root=data_dir,
        test_envs=test_envs,
        hparams=hparams,
        overlap_type=overlap_type,
        num_classes=num_classes,
        num_domain_linked_classes=num_domain_linked_classes,
    )
    hparams["C_oc"] = dataset.overlapping_classes
    logging.info(f"Loaded {dataset_name}")

    # ----------------------------
    # SPLIT ENVIRONMENTS
    # ----------------------------
    in_splits, out_splits = [], []
    relative_test_env = None
    log_dir = logger.return_root()

    for env_i, env in enumerate(dataset):
        out, in_ = split_dataset(
            env,
            int(len(env) * holdout_fraction),
            seed_hash(trial_seed, env_i)
        )

        if hparams.get("class_balanced", False):
            in_weights = make_weights_for_balanced_classes(in_)
            out_weights = make_weights_for_balanced_classes(out)
        else:
            in_weights, out_weights = None, None

        in_splits.append((in_, in_weights))
        out_splits.append((out, out_weights))

        if env.is_test_env:
            relative_test_env = env_i

    assert relative_test_env is not None, "No testing domains"
    logging.info(f"test_envs={test_envs}, relative_test_env={relative_test_env}")

    # ----------------------------
    # DATA LOADERS
    # ----------------------------
    train_loaders = [
        InfiniteDataLoader(
            dataset=env,
            weights=env_weights,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for i, (env, env_weights) in enumerate(in_splits)
        if i != relative_test_env
    ]

    eval_loaders = [
        FastDataLoader(
            dataset=env,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for env, _ in (in_splits + out_splits)
    ]

    eval_weights = [None for _ in (in_splits + out_splits)]

    eval_loader_names = (
        [f"env{i}_in" for i in range(len(in_splits))] +
        [f"env{i}_out" for i in range(len(out_splits))]
    )

    logging.info(f"Created data loaders: {eval_loader_names}")

    train_minibatches_iterator = zip(*train_loaders)
    steps_per_epoch = min(
        [len(env) / hparams["batch_size"] for env, _ in in_splits]
    )

    # ----------------------------
    # INITIALIZE ALGORITHM
    # ----------------------------
    algorithm = ALGORITHMS[algorithm_name](
        input_shape=dataset.input_shape,
        num_classes=dataset.num_classes,
        num_domains=len(dataset) - len(test_envs),
        hparams=hparams,
    )
    algorithm.to(device)
    logging.info(f"Algorithm {algorithm_name} initialized")

    # ----------------------------
    # MODEL STATISTICS
    # ----------------------------
    total_params = sum(p.numel() for p in algorithm.parameters())
    trainable_params = sum(p.numel() for p in algorithm.parameters() if p.requires_grad)
    model_size_mb = sum(
        p.nelement() * p.element_size() for p in algorithm.parameters()
    ) / (1024 ** 2)

    logging.info(f"Total params: {total_params:,}")
    logging.info(f"Trainable params: {trainable_params:,}")
    logging.info(f"Model size: {model_size_mb:.2f} MB")

    # ----------------------------
    # SWAD INITIALIZATION
    # ----------------------------
    if use_swad:
        swad_weights = {
            name: p.clone().detach()
            for name, p in algorithm.named_parameters()
            if p.requires_grad
        }
        swad_n = 0

    # ----------------------------
    # TRAINING LOOP
    # ----------------------------
    checkpoint_vals = collections.defaultdict(list)
    training_start_time = time.time()
    epoch_start_time = time.time()
    current_epoch = 0
    steps_in_current_epoch = 0

    for step in tqdm(range(n_steps)):
        minibatches_device = [
            (x.to(device), y.to(device))
            for x, y in next(train_minibatches_iterator)
        ]

        step_vals = algorithm.update(minibatches_device, None)
        for k, v in step_vals.items():
            checkpoint_vals[k].append(v)

        # ---- SWAD UPDATE ----
        if use_swad and current_epoch >= swad_start_epoch:
            swad_n += 1
            for name, p in algorithm.named_parameters():
                if p.requires_grad:
                    swad_weights[name] = (
                        (swad_n - 1) * swad_weights[name] + p.detach()
                    ) / swad_n

        # ---- Epoch tracking ----
        steps_in_current_epoch += 1
        if steps_in_current_epoch >= steps_per_epoch:
            current_epoch += 1
            steps_in_current_epoch = 0
            epoch_start_time = time.time()

        # ---- Evaluation ----
        if step % checkpoint_freq == 0 or step == n_steps - 1:
            if use_swad and swad_n > 0:
                original_params = {
                    n: p.clone() for n, p in algorithm.named_parameters()
                }
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(swad_weights[n])

            results_dict = {
                key: {
                    _key: []
                    for _key in ["acc", "f1", "nacc", "oacc", "recall", "precision", "loss"]
                }
                for key in ["train", "val", "test", "other"]
            }

            # Calculate training value averages
            for key, val in checkpoint_vals.items():
                results_dict["train"][str(key)] = [np.mean(val)]

            # Evaluation
            for name, loader, weights in zip(
                eval_loader_names, eval_loaders, eval_weights
            ):
                (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
                    algorithm, loader, weights, device, dataset
                )
                loss = compute_loss(algorithm, loader, device)

                domain_idx = int(name[3])

                if domain_idx == relative_test_env:
                    if "in" in name:
                        loader_type = "test"
                    else:
                        loader_type = "other"
                elif "out" in name:
                    loader_type = "val"
                else:
                    loader_type = "train"

                results_dict[loader_type]["acc"].append(float(acc))
                results_dict[loader_type]["recall"].append(float(recall))
                results_dict[loader_type]["f1"].append(float(f1))
                results_dict[loader_type]["precision"].append(float(precision))
                results_dict[loader_type]["nacc"].append(float(nacc))
                results_dict[loader_type]["oacc"].append(float(oacc))
                results_dict[loader_type]["loss"].append(float(loss))

            # Log metrics
            current_epoch_num = step / steps_per_epoch
            print(f"\n=== Step {step} (Epoch {current_epoch_num:.2f}) ===")

            for stage in ["train", "val", "test", "other"]:
                if results_dict[stage]["acc"]:
                    print(f"{stage.upper()}:")
                    print(f"  loss: {np.mean(results_dict[stage]['loss']):.4f}")
                    print(f"  acc: {np.mean(results_dict[stage]['acc']):.4f}")
                    print(f"  precision: {np.mean(results_dict[stage]['precision']):.4f}")
                    print(f"  recall: {np.mean(results_dict[stage]['recall']):.4f}")
                    print(f"  f1: {np.mean(results_dict[stage]['f1']):.4f}")
                    print(f"  oacc: {np.mean(results_dict[stage]['oacc']):.4f}")
                    print(f"  nacc: {np.mean(results_dict[stage]['nacc']):.4f}")

            logger.log(results_dict, step)
            checkpoint_vals.clear()

            if use_swad and swad_n > 0:
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(original_params[n])

    # ----------------------------
    # TRAINING COMPLETION SUMMARY
    # ----------------------------
    total_training_time = time.time() - training_start_time
    logging.info("=" * 80)
    logging.info("TRAINING COMPLETED")
    logging.info("=" * 80)
    logging.info(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    logging.info(f"Total steps: {n_steps}")
    logging.info(f"Total epochs: {current_epoch}")
    logging.info("=" * 80)

    print("Training completed.")

In [ ]:
print("BEST_HPARAMS:", BEST_HPARAMS)


In [ ]:
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        # Map env keys to train/val/test safely
        mapping = {}
        for key in results_dict.keys():
            if "in" in key:
                mapping[key] = "train"
            elif "out" in key:
                mapping[key] = "val"
            else:
                mapping[key] = "test"

        print(f"\nStep {step} metrics:")
        for k, split in mapping.items():
            vals = results_dict[k]

            # Use len() to safely check
            acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
            loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None

            if acc is not None and loss is not None:
                print(f"  {split.upper()} ({k}) | Acc: {acc:.4f} | Loss: {loss:.4f}")


In [ ]:
import os
from pathlib import Path

# ----------------------------
# Experiment directory
# ----------------------------
EXP_DIR = "/kaggle/working/pacs_experiment"  # change to pacs or vlcs
os.makedirs(EXP_DIR, exist_ok=True)

# ----------------------------
# Logger (simple print logger or CSV logger)
# ----------------------------
class PrintLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"Step {step} metrics:")
        for split in ["train", "val", "test"]:
            if split in results_dict and len(results_dict[split]["acc"]) > 0:
                print(f"  {split.upper()} | Acc: {np.mean(results_dict[split]['acc']):.4f} | Loss: {np.mean(results_dict[split]['loss']):.4f}")

logger = SafeLogger()


In [ ]:
import os
from pathlib import Path
import logging
import numpy as np
import torch

# ----------------------------
# Experiment directory
# ----------------------------
OUTPUT_DIR = "/kaggle/working/experiments"
EXP_NAME = "pacs_erm_best_hparams"  # updated for PACS
EXP_DIR = Path(OUTPUT_DIR) / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = EXP_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO)
logging.info(f"Experiment directory: {EXP_DIR}")

# ----------------------------
# Safe Logger
# ----------------------------
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"\nStep {step} metrics:")
        for split in ["train", "val", "test"]:
            if split in results_dict:
                vals = results_dict[split]
                acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
                loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None
                if acc is not None and loss is not None:
                    print(f"  {split.upper()} | Acc: {acc:.4f} | Loss: {loss:.4f}")

logger = SafeLogger()

# ----------------------------
# Use Optuna-tuned hyperparameters for PACS
# ----------------------------
BEST_HPARAMS = {
    'lr': 1.3332198891320255e-05,
    'batch_size': 16,
    'weight_decay': 9.478198291385849e-05,
    'optimizer': 'Adam',
    'resnet_dropout': 0.33172328324192074,
    'lr_scheduler': 'StepLR'
}

# ----------------------------
# Correct overlap_type for PACS
# ----------------------------
VALID_OVERLAP_TYPE = "high"  # match PACS dataset config

# ----------------------------
# Call fit_simple to train full PACS dataset
# ----------------------------
fit_simple(
    exp_dir=EXP_DIR,
    logger=logger,
    seed=42,
    trial_seed=0,
    hparams_seed=-1,           # use BEST_HPARAMS
    algorithm_name="ERM",
    dataset_name="PACS",
    data_dir="/kaggle/input/pacs-dataset/kfold",  # PACS dataset path
    num_workers=12,
    test_envs=[0],             # e.g., Art domain as test
    overlap_type=VALID_OVERLAP_TYPE,
    holdout_fraction=0.2,
    n_steps=5001,
    checkpoint_freq=300,
    use_swad=True,
    swad_start_epoch=1
)

print("\n✅ Training of full PACS dataset completed successfully.")


## Hyperparameter Tuning of VLCS using Optuna

In [ ]:
dataset_name_vlcs = config["dataset"][1]
dataset_name_vlcs

In [ ]:
# ===== Imports and Setup for VLCS =====
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, models, datasets
import optuna
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ===== Seeds for reproducibility =====
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# ===== Device setup =====
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ===== Standard transform for ResNet input =====
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split, DataLoader
from torchvision import transforms

# ===== VLCS dataset path =====
DATA_DIR_VLCS = "/kaggle/input/vlcsdataset/VLCS"  # Change this to your VLCS folder path

# Standard transform for ResNet input
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def load_vlcs_dataset(split="train", val_ratio=0.2, seed=0):
    """
    Load VLCS dataset and split into train/val.
    VLCS classes typically: Caltech, LabelMe, SUN, VOC
    """
    full_dataset = ImageFolder(root=DATA_DIR_VLCS, transform=base_transform)

    total_len = len(full_dataset)
    val_len = int(total_len * val_ratio)
    train_len = total_len - val_len

    generator = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len], generator=generator)

    if split == "train":
        return train_dataset
    else:
        return val_dataset


In [ ]:
def objective(trial):
    # --- Hyperparameters ---
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    scheduler_name = trial.suggest_categorical("scheduler", ["StepLR", "CosineAnnealingLR"])

    # --- Load VLCS datasets ---
    train_dataset = load_vlcs_dataset(split="train")
    val_dataset   = load_vlcs_dataset(split="val")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # --- Model ---
    model = models.resnet18(pretrained=True)
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, 5)  # VLCS has 5 classes if merged domains, adjust if needed
    )
    model = model.to(device)

    # --- Optimizer ---
    if optimizer_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    # --- Scheduler ---
    if scheduler_name == "StepLR":
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    else:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    # --- Loss ---
    criterion = nn.CrossEntropyLoss()

    # --- Training loop (small for tuning) ---
    best_val_acc = 0
    for epoch in range(3):  # small number of epochs for tuning
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(x)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                y_hat = model(x)
                correct += (y_hat.argmax(1) == y).sum().item()
                total += y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc

    return best_val_acc


In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=15)

print("Best Hyperparameters:")
print(study.best_params)



In [ ]:
# --- Save best hyperparameters from Optuna ---
best = study.best_params
print("Best Hyperparameters:", best)

# Example output format:
# {'lr': 0.000623, 'batch_size': 64, 'optimizer': 'SGD',
#  'weight_decay': 0.00668, 'aug_strength': 0.0713,
#  'scheduler': 'CosineAnnealingLR', 'dropout': 0.118}


In [ ]:
# --- Load VLCS datasets using best hyperparameters ---
train_dataset = load_vlcs_dataset(split="train")
val_dataset   = load_vlcs_dataset(split="val")
test_dataset  = val_dataset  # Use separate test set if you don't have a separate test split

train_loader = DataLoader(train_dataset, batch_size=best["batch_size"], shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=best["batch_size"], shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=best["batch_size"], shuffle=False, num_workers=4)


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# --- VLCS has 4 classes/domains ---
num_classes = 4  # caltech, labelme, sun09, voc2007

# --- Define model ---
model = models.resnet18(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(best["dropout"]),
    nn.Linear(model.fc.in_features, num_classes)
)
model = model.to(device)

# --- Optimizer ---
if best["optimizer"] == "Adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=best["lr"], momentum=0.9, weight_decay=best["weight_decay"])

# --- Scheduler ---
if best["scheduler"] == "StepLR":
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
else:
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# --- Loss function ---
criterion = nn.CrossEntropyLoss()


In [ ]:
train_acc_list, val_acc_list, test_acc_list = [], [], []
train_loss_list, val_loss_list, test_loss_list = [], [], []

num_epochs = 10
for epoch in range(num_epochs):
    # ===== TRAIN =====
    model.train()
    running_loss, correct, total = 0, 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_hat = model(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * y.size(0)
        correct += (y_hat.argmax(1) == y).sum().item()
        total += y.size(0)
    train_loss = running_loss / total
    train_acc = correct / total
    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc)
    scheduler.step()

    # ===== VALIDATION =====
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(1) == y).sum().item()
            total += y.size(0)
    val_loss = running_loss / total
    val_acc = correct / total
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)

    # ===== TEST =====
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for x, y in test_loader:  # SUN09 or whichever domain is test
            x, y = x.to(device), y.to(device)
            y_hat = model(x)
            loss = criterion(y_hat, y)
            running_loss += loss.item() * y.size(0)
            correct += (y_hat.argmax(1) == y).sum().item()
            total += y.size(0)
    test_loss = running_loss / total
    test_acc = correct / total
    test_loss_list.append(test_loss)
    test_acc_list.append(test_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f} | "
          f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f}")


In [ ]:
import matplotlib.pyplot as plt

# ---- Loss Plot ----
plt.figure(figsize=(10,5))
plt.plot(train_loss_list, label="Train Loss", marker='o')
plt.plot(val_loss_list, label="Validation Loss", marker='s')
plt.plot(test_loss_list, label="Test Loss", marker='^')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("VLCS Train / Validation / Test Loss")
plt.legend()
plt.grid(True)
plt.show()

# ---- Accuracy Plot ----
plt.figure(figsize=(10,5))
plt.plot(train_acc_list, label="Train Accuracy", marker='o')
plt.plot(val_acc_list, label="Validation Accuracy", marker='s')
plt.plot(test_acc_list, label="Test Accuracy", marker='^')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("VLCS Train / Validation / Test Accuracy")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# ===== Collect predictions =====
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        preds = y_hat.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# ===== Classification Report =====
print("VLCS Classification Report:")
print(classification_report(all_labels, all_preds, digits=4))

# ===== Confusion Matrix =====
cm = confusion_matrix(all_labels, all_preds)

# VLCS domain names (ImageFolder alphabetical order)
class_names = ["Caltech", "LabelMe", "SUN09", "VOC2007"]

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("VLCS Confusion Matrix")
plt.tight_layout()
plt.show()


# Training with Best Hyperparameters for VLCS

In [ ]:
# =========================================================
# Inject Optuna best hyperparameters into fit_simple (VLCS)
# =========================================================

BEST_HPARAMS = {
    "lr": 0.0004265412926501881,
    "batch_size": 32,
    "weight_decay": 3.0112348644815128e-05,
    "optimizer": "SGD",
    "resnet_dropout": 0.3802762000352884,
    "lr_scheduler": "StepLR"
}

print("Using Optuna-tuned hyperparameters for VLCS:")
for k, v in BEST_HPARAMS.items():
    print(f"  {k}: {v}")


In [ ]:
def fit_simple(
    exp_dir: str,
    logger,  # CSVLogger or PrintLogger
    seed: int,
    trial_seed: int,
    hparams_seed: int,
    algorithm_name: str,
    dataset_name: str,
    data_dir: str,
    num_workers: int,
    test_envs: list,
    overlap_type: str,
    holdout_fraction: float = 0.2,
    n_steps: int = 5001,
    checkpoint_freq: int = 300,
    model_checkpoint=None,
    teacher_paths=None,
    num_domain_linked_classes=None,
    num_classes=None,
    auto_augment: bool = False,
    augment_search_epochs: int = 10,
    use_swad: bool = True,
    swad_start_epoch: int = 1
):
    import collections, os, json, time, logging
    import numpy as np
    import torch
    from tqdm import tqdm
    import matplotlib.pyplot as plt
    import seaborn as sns

    # ----------------------------
    # SEED AND DEVICE
    # ----------------------------
    L.seed_everything(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ----------------------------
    # HYPERPARAMETERS
    # ----------------------------
    if hparams_seed == -1:
        # === OPTUNA-TUNED HYPERPARAMETERS ===
        hparams = default_hparams(algorithm_name, dataset_name)
        hparams.update(BEST_HPARAMS)
        logging.info("Using Optuna-tuned hyperparameters")
    else:
        if hparams_seed == 0:
            hparams = default_hparams(algorithm_name, dataset_name)
        else:
            hparams = random_hparams(
                algorithm_name,
                dataset_name,
                seed_hash(hparams_seed, trial_seed)
            )

    logging.info(f"hparams: {hparams}")

    # ----------------------------
    # LOAD DATASET
    # ----------------------------
    dataset = DATASETS[dataset_name](
        root=data_dir,
        test_envs=test_envs,
        hparams=hparams,
        overlap_type=overlap_type,
        num_classes=num_classes,
        num_domain_linked_classes=num_domain_linked_classes,
    )
    hparams["C_oc"] = dataset.overlapping_classes
    logging.info(f"Loaded {dataset_name}")

    # ----------------------------
    # SPLIT ENVIRONMENTS
    # ----------------------------
    in_splits, out_splits = [], []
    relative_test_env = None
    log_dir = logger.return_root()

    for env_i, env in enumerate(dataset):
        out, in_ = split_dataset(
            env,
            int(len(env) * holdout_fraction),
            seed_hash(trial_seed, env_i)
        )

        if hparams.get("class_balanced", False):
            in_weights = make_weights_for_balanced_classes(in_)
            out_weights = make_weights_for_balanced_classes(out)
        else:
            in_weights, out_weights = None, None

        in_splits.append((in_, in_weights))
        out_splits.append((out, out_weights))

        if env.is_test_env:
            relative_test_env = env_i

    assert relative_test_env is not None, "No testing domains"
    logging.info(f"test_envs={test_envs}, relative_test_env={relative_test_env}")

    # ----------------------------
    # DATA LOADERS
    # ----------------------------
    train_loaders = [
        InfiniteDataLoader(
            dataset=env,
            weights=env_weights,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for i, (env, env_weights) in enumerate(in_splits)
        if i != relative_test_env
    ]

    eval_loaders = [
        FastDataLoader(
            dataset=env,
            batch_size=hparams["batch_size"],
            num_workers=num_workers
        )
        for env, _ in (in_splits + out_splits)
    ]

    eval_weights = [None for _ in (in_splits + out_splits)]

    eval_loader_names = (
        [f"env{i}_in" for i in range(len(in_splits))] +
        [f"env{i}_out" for i in range(len(out_splits))]
    )

    logging.info(f"Created data loaders: {eval_loader_names}")

    train_minibatches_iterator = zip(*train_loaders)
    steps_per_epoch = min(
        [len(env) / hparams["batch_size"] for env, _ in in_splits]
    )

    # ----------------------------
    # INITIALIZE ALGORITHM
    # ----------------------------
    algorithm = ALGORITHMS[algorithm_name](
        input_shape=dataset.input_shape,
        num_classes=dataset.num_classes,
        num_domains=len(dataset) - len(test_envs),
        hparams=hparams,
    )
    algorithm.to(device)
    logging.info(f"Algorithm {algorithm_name} initialized")

    # ----------------------------
    # MODEL STATISTICS
    # ----------------------------
    total_params = sum(p.numel() for p in algorithm.parameters())
    trainable_params = sum(p.numel() for p in algorithm.parameters() if p.requires_grad)
    model_size_mb = sum(
        p.nelement() * p.element_size() for p in algorithm.parameters()
    ) / (1024 ** 2)

    logging.info(f"Total params: {total_params:,}")
    logging.info(f"Trainable params: {trainable_params:,}")
    logging.info(f"Model size: {model_size_mb:.2f} MB")

    # ----------------------------
    # SWAD INITIALIZATION
    # ----------------------------
    if use_swad:
        swad_weights = {
            name: p.clone().detach()
            for name, p in algorithm.named_parameters()
            if p.requires_grad
        }
        swad_n = 0

    # ----------------------------
    # TRAINING LOOP
    # ----------------------------
    checkpoint_vals = collections.defaultdict(list)
    training_start_time = time.time()
    epoch_start_time = time.time()
    current_epoch = 0
    steps_in_current_epoch = 0

    for step in tqdm(range(n_steps)):
        minibatches_device = [
            (x.to(device), y.to(device))
            for x, y in next(train_minibatches_iterator)
        ]

        step_vals = algorithm.update(minibatches_device, None)
        for k, v in step_vals.items():
            checkpoint_vals[k].append(v)

        # ---- SWAD UPDATE ----
        if use_swad and current_epoch >= swad_start_epoch:
            swad_n += 1
            for name, p in algorithm.named_parameters():
                if p.requires_grad:
                    swad_weights[name] = (
                        (swad_n - 1) * swad_weights[name] + p.detach()
                    ) / swad_n

        # ---- Epoch tracking ----
        steps_in_current_epoch += 1
        if steps_in_current_epoch >= steps_per_epoch:
            current_epoch += 1
            steps_in_current_epoch = 0
            epoch_start_time = time.time()

        # ---- Evaluation ----
        if step % checkpoint_freq == 0 or step == n_steps - 1:
            if use_swad and swad_n > 0:
                original_params = {
                    n: p.clone() for n, p in algorithm.named_parameters()
                }
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(swad_weights[n])

            results_dict = {
                key: {
                    _key: []
                    for _key in ["acc", "f1", "nacc", "oacc", "recall", "precision", "loss"]
                }
                for key in ["train", "val", "test", "other"]
            }

            # Calculate training value averages
            for key, val in checkpoint_vals.items():
                results_dict["train"][str(key)] = [np.mean(val)]

            # Evaluation
            for name, loader, weights in zip(
                eval_loader_names, eval_loaders, eval_weights
            ):
                (acc, recall, f1, precision, oacc, nacc, per_class_acc, cm) = accuracy(
                    algorithm, loader, weights, device, dataset
                )
                loss = compute_loss(algorithm, loader, device)

                domain_idx = int(name[3])

                if domain_idx == relative_test_env:
                    if "in" in name:
                        loader_type = "test"
                    else:
                        loader_type = "other"
                elif "out" in name:
                    loader_type = "val"
                else:
                    loader_type = "train"

                results_dict[loader_type]["acc"].append(float(acc))
                results_dict[loader_type]["recall"].append(float(recall))
                results_dict[loader_type]["f1"].append(float(f1))
                results_dict[loader_type]["precision"].append(float(precision))
                results_dict[loader_type]["nacc"].append(float(nacc))
                results_dict[loader_type]["oacc"].append(float(oacc))
                results_dict[loader_type]["loss"].append(float(loss))

            # Log metrics
            current_epoch_num = step / steps_per_epoch
            print(f"\n=== Step {step} (Epoch {current_epoch_num:.2f}) ===")

            for stage in ["train", "val", "test", "other"]:
                if results_dict[stage]["acc"]:
                    print(f"{stage.upper()}:")
                    print(f"  loss: {np.mean(results_dict[stage]['loss']):.4f}")
                    print(f"  acc: {np.mean(results_dict[stage]['acc']):.4f}")
                    print(f"  precision: {np.mean(results_dict[stage]['precision']):.4f}")
                    print(f"  recall: {np.mean(results_dict[stage]['recall']):.4f}")
                    print(f"  f1: {np.mean(results_dict[stage]['f1']):.4f}")
                    print(f"  oacc: {np.mean(results_dict[stage]['oacc']):.4f}")
                    print(f"  nacc: {np.mean(results_dict[stage]['nacc']):.4f}")

            logger.log(results_dict, step)
            checkpoint_vals.clear()

            if use_swad and swad_n > 0:
                for n, p in algorithm.named_parameters():
                    if p.requires_grad:
                        p.data.copy_(original_params[n])

    # ----------------------------
    # TRAINING COMPLETION SUMMARY
    # ----------------------------
    total_training_time = time.time() - training_start_time
    logging.info("=" * 80)
    logging.info("TRAINING COMPLETED")
    logging.info("=" * 80)
    logging.info(f"Total training time: {total_training_time:.2f} seconds ({total_training_time/60:.2f} minutes)")
    logging.info(f"Total steps: {n_steps}")
    logging.info(f"Total epochs: {current_epoch}")
    logging.info("=" * 80)

    print("Training completed.")

In [ ]:
print("BEST_HPARAMS:", BEST_HPARAMS)


In [ ]:
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        # Map env keys to train/val/test safely
        mapping = {}
        for key in results_dict.keys():
            if "in" in key:
                mapping[key] = "train"
            elif "out" in key:
                mapping[key] = "val"
            else:
                mapping[key] = "test"

        print(f"\nStep {step} metrics:")
        for k, split in mapping.items():
            vals = results_dict[k]

            # Use len() to safely check
            acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
            loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None

            if acc is not None and loss is not None:
                print(f"  {split.upper()} ({k}) | Acc: {acc:.4f} | Loss: {loss:.4f}")


In [ ]:
import os
from pathlib import Path
import numpy as np

# ----------------------------
# Experiment directory for VLCS
# ----------------------------
EXP_DIR = "/kaggle/working/vlcs_experiment"
os.makedirs(EXP_DIR, exist_ok=True)

# ----------------------------
# Logger (simple print logger or CSV logger)
# ----------------------------
class PrintLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"Step {step} metrics:")
        for split in ["train", "val", "test"]:
            if split in results_dict and results_dict[split]["acc"]:
                print(f"  {split.upper()} | Acc: {np.mean(results_dict[split]['acc']):.4f} | Loss: {np.mean(results_dict[split]['loss']):.4f}")

logger = SafeLogger()


In [ ]:
import os
from pathlib import Path
import logging
import numpy as np
import torch

# ----------------------------
# Experiment directory
# ----------------------------
OUTPUT_DIR = "/kaggle/working/experiments"
EXP_NAME = "vlcs_erm_best_hparams"
EXP_DIR = Path(OUTPUT_DIR) / EXP_NAME
EXP_DIR.mkdir(parents=True, exist_ok=True)

LOG_DIR = EXP_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO)
logging.info(f"Experiment directory: {EXP_DIR}")

# ----------------------------
# Safe Logger
# ----------------------------
class SafeLogger:
    def return_root(self):
        return EXP_DIR

    def log(self, results_dict, step):
        print(f"\nStep {step} metrics:")
        for split in ["train", "val", "test"]:
            if split in results_dict:
                vals = results_dict[split]
                # Safe check using len()
                acc = np.mean(vals["acc"]) if "acc" in vals and len(vals["acc"]) > 0 else None
                loss = np.mean(vals["loss"]) if "loss" in vals and len(vals["loss"]) > 0 else None
                if acc is not None and loss is not None:
                    print(f"  {split.upper()} | Acc: {acc:.4f} | Loss: {loss:.4f}")

logger = SafeLogger()

# ----------------------------
# Use Optuna-tuned hyperparameters for VLCS
# ----------------------------
BEST_HPARAMS = {
    'lr': 0.0004265412926501881,
    'batch_size': 32,
    'weight_decay': 3.0112348644815128e-05,
    'optimizer': 'SGD',
    'resnet_dropout': 0.3802762000352884,
    'lr_scheduler': 'StepLR'
}

# ----------------------------
# Overlap type for VLCS
# ----------------------------
VALID_OVERLAP_TYPE = "high"  # adjust if needed

# ----------------------------
# Call fit_simple to train full VLCS dataset
# ----------------------------
fit_simple(
    exp_dir=EXP_DIR,
    logger=logger,
    seed=42,                   # reproducible
    trial_seed=0,
    hparams_seed=-1,           # use BEST_HPARAMS
    algorithm_name="ERM",      # your algorithm
    dataset_name="VLCS",
    data_dir="/kaggle/input/vlcsdataset",
    num_workers=12,
    test_envs=[0],             # Caltech as test
    overlap_type=VALID_OVERLAP_TYPE,
    holdout_fraction=0.2,
    n_steps=5001,
    checkpoint_freq=300,
    use_swad=True,
    swad_start_epoch=1
)

print("\n✅ Training of full VLCS dataset completed successfully.")
